In [1]:
import pandas as pd, numpy as np, warnings, json
warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                              ConfusionMatrixDisplay, classification_report)
from scipy import stats
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ============================================================
# 1. LOAD + CLEAN
# ============================================================
df = pd.read_csv('/kaggle/input/datasets/sham1026/indian-liver-patient-dataset/Indian Liver Patient Dataset (ILPD).csv')
df.columns = ['age','gender','total_bilirubin','direct_bilirubin','alkaline_phosphotase',
              'alt_sgpt','ast_sgot','total_proteins','albumin','ag_ratio','target']

n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
n_dup = n0 - len(df)

df['gender'] = (df['gender'] == 'Male').astype(int)
df['target'] = (df['target'] == 1).astype(int)     # 1 = liver patient, 0 = healthy

df.to_csv('ILPD_cleaned.csv', index=False)

log_report = []
log_report.append(f"Raw rows: {n0}")
log_report.append(f"Duplicate rows removed: {n_dup}")
log_report.append(f"Final rows: {len(df)}")
log_report.append(f"Missing values (before impute):\n{df.isna().sum()[df.isna().sum()>0].to_string()}")
log_report.append(f"Class distribution:\n{df['target'].value_counts().to_string()}  (1=patient, 0=healthy)")
log_report.append(f"Class balance ratio: {df['target'].mean():.3f} positive")

# ============================================================
# 2. EDA
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df['target'].map({1: 'Liver Patient', 0: 'Healthy'}).value_counts().plot.bar(
    ax=axes[0], color=['#c0392b', '#2471a3'])
axes[0].set_title('Class Distribution'); axes[0].set_ylabel('Count'); axes[0].tick_params(rotation=0)
df['age'].plot.hist(ax=axes[1], bins=20, color='#1a3a6b', edgecolor='white')
axes[1].set_title('Age Distribution'); axes[1].set_xlabel('Age')
plt.tight_layout(); plt.savefig('eda_class_age.png', dpi=150); plt.close()

plt.figure(figsize=(7, 6))
corr = df.drop(columns='target').corr()
im = plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046)
plt.xticks(range(len(corr)), corr.columns, rotation=90, fontsize=8)
plt.yticks(range(len(corr)), corr.columns, fontsize=8)
for i in range(len(corr)):
    for j in range(len(corr)):
        plt.text(j, i, f"{corr.iloc[i,j]:.2f}", ha='center', va='center', fontsize=6)
plt.title('Feature Correlation Heatmap')
plt.tight_layout(); plt.savefig('eda_correlation.png', dpi=150); plt.close()

skew_cols = ['total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase', 'alt_sgpt', 'ast_sgot']
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df[skew_cols].skew().plot.bar(ax=axes[0], color='#c0392b')
axes[0].set_title('Skewness BEFORE log1p transform'); axes[0].tick_params(rotation=45)
np.log1p(df[skew_cols]).skew().plot.bar(ax=axes[1], color='#27ae60')
axes[1].set_title('Skewness AFTER log1p transform'); axes[1].tick_params(rotation=45)
plt.tight_layout(); plt.savefig('eda_skewness.png', dpi=150); plt.close()

# ============================================================
# 3. PREPROCESSING PIPELINE (leakage-safe: fit inside CV only)
# ============================================================
X = df.drop(columns='target')
y = df['target']

def make_preprocessor():
    return ColumnTransformer([
        ('log', Pipeline([('imp', SimpleImputer(strategy='median')),
                           ('l', FunctionTransformer(np.log1p))]), skew_cols),
        ('num', SimpleImputer(strategy='median'), ['age', 'total_proteins', 'albumin', 'ag_ratio']),
        ('bin', 'passthrough', ['gender'])
    ])

def make_pipeline(model):
    return Pipeline([('prep', make_preprocessor()), ('sc', StandardScaler()), ('m', model)])

# ============================================================
# 4. MODELS — 6-way comparison
# ============================================================
outer_cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE + 1)
scoring = {'auc': 'roc_auc', 'acc': 'accuracy', 'rec': 'recall', 'prec': 'precision',
           'f1': 'f1', 'bal': 'balanced_accuracy'}

models = {
    'Logistic Regression': make_pipeline(LogisticRegression(max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE)),
    'K-Nearest Neighbors': GridSearchCV(make_pipeline(KNeighborsClassifier()),
        {'m__n_neighbors': [5, 9, 15, 21], 'm__weights': ['uniform', 'distance']},
        scoring='roc_auc', cv=inner_cv),
    'Naive Bayes': make_pipeline(GaussianNB()),
    'SVM (RBF Kernel)': GridSearchCV(make_pipeline(SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE)),
        {'m__C': [0.1, 1, 10, 50], 'm__gamma': ['scale', 0.01, 0.1]},
        scoring='roc_auc', cv=inner_cv),
    'Random Forest': GridSearchCV(make_pipeline(RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE)),
        {'m__n_estimators': [200, 400], 'm__max_depth': [4, 6, None], 'm__min_samples_leaf': [2, 5]},
        scoring='roc_auc', cv=inner_cv),
    'Gradient Boosting': GridSearchCV(make_pipeline(GradientBoostingClassifier(random_state=RANDOM_STATE)),
        {'m__n_estimators': [100, 200], 'm__max_depth': [2, 3], 'm__learning_rate': [0.05, 0.1]},
        scoring='roc_auc', cv=inner_cv),
}

rows = []
fold_auc_scores = {}   # for statistical significance testing
oof_probs = {}          # out-of-fold predicted probabilities, for ROC plot

for name, mdl in models.items():
    res = cross_validate(mdl, X, y, cv=outer_cv, scoring=scoring, return_estimator=False)
    fold_auc_scores[name] = res['test_auc']
    rows.append({'Model': name, **{k: f"{res['test_'+k].mean():.3f} ± {res['test_'+k].std():.3f}" for k in scoring}})
    oof_probs[name] = cross_val_predict(mdl, X, y, cv=outer_cv, method='predict_proba')[:, 1]

results_df = pd.DataFrame(rows).sort_values('Model')
results_df.to_csv('unimodal_results_table.csv', index=False)
print(results_df.to_string(index=False))

# ============================================================
# 5. STATISTICAL SIGNIFICANCE: paired t-test between top-2 models
# ============================================================
mean_auc = {k: v.mean() for k, v in fold_auc_scores.items()}
ranked = sorted(mean_auc.items(), key=lambda x: -x[1])
best_name, second_name = ranked[0][0], ranked[1][0]
t_stat, p_val = stats.ttest_rel(fold_auc_scores[best_name], fold_auc_scores[second_name])
sig_report = (f"Best model: {best_name} (mean AUC={ranked[0][1]:.3f})\n"
              f"2nd best  : {second_name} (mean AUC={ranked[1][1]:.3f})\n"
              f"Paired t-test on per-fold AUC ({best_name} vs {second_name}): "
              f"t={t_stat:.3f}, p={p_val:.3f} "
              f"({'statistically significant (p<0.05)' if p_val < 0.05 else 'NOT statistically significant (p>=0.05)'})")
print('\n' + sig_report)

# ============================================================
# 6. ROC CURVES — all 6 models
# ============================================================
plt.figure(figsize=(6, 5.5))
colors_map = plt.cm.tab10(np.linspace(0, 1, len(models)))
for (name, probs), c in zip(oof_probs.items(), colors_map):
    fpr, tpr, _ = roc_curve(y, probs)
    auc_val = roc_auc_score(y, probs)
    lw = 2.5 if name == best_name else 1.3
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.3f})", color=c, linewidth=lw)
plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random (AUC=0.50)')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Unimodal Baseline\n(5-fold CV, out-of-fold predictions)')
plt.legend(fontsize=8, loc='lower right')
plt.tight_layout(); plt.savefig('unimodal_roc_all_models.png', dpi=150); plt.close()

# ============================================================
# 7. CONFUSION MATRIX for best model
# ============================================================
best_preds = (oof_probs[best_name] >= 0.5).astype(int)
cm = confusion_matrix(y, best_preds)
plt.figure(figsize=(4.5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=['Healthy', 'Liver Patient'])
disp.plot(cmap='Blues', values_format='d', colorbar=False)
plt.title(f'Confusion Matrix — {best_name}\n(out-of-fold, threshold=0.5)')
plt.tight_layout(); plt.savefig('unimodal_confusion_matrix.png', dpi=150); plt.close()

class_report_str = classification_report(y, best_preds, target_names=['Healthy', 'Liver Patient'])
print('\nClassification report for best model:\n', class_report_str)

# ============================================================
# 8. FEATURE IMPORTANCE (Random Forest, refit on full data)
# ============================================================
rf_search = models['Random Forest']
rf_search.fit(X, y)
best_rf = rf_search.best_estimator_
importances = pd.Series(
    best_rf.named_steps['m'].feature_importances_,
    index=skew_cols + ['age', 'total_proteins', 'albumin', 'ag_ratio', 'gender']
).sort_values()
plt.figure(figsize=(5.5, 4.2))
importances.plot.barh(color='#1a3a6b')
plt.title('Random Forest Feature Importance (full-data refit)')
plt.xlabel('Importance'); plt.tight_layout(); plt.savefig('unimodal_feature_importance.png', dpi=150); plt.close()

# ============================================================
# 9. FINALISE: refit BEST model pipeline on full data & save
# ============================================================
best_model_search = models[best_name]
best_model_search.fit(X, y)
final_model = best_model_search.best_estimator_ if hasattr(best_model_search, 'best_estimator_') else best_model_search
joblib.dump(final_model, 'unimodal_best_model.joblib')
best_params = best_model_search.best_params_ if hasattr(best_model_search, 'best_params_') else 'N/A (no grid search)'

# ============================================================
# 10. WRITE FULL TEXT REPORT
# ============================================================
with open('unimodal_final_report.txt', 'w') as f:
    f.write("="*70 + "\n")
    f.write("UNIMODAL BASELINE — FINAL REPORT (ILPD, tabular clinical/lab data)\n")
    f.write("="*70 + "\n\n")
    f.write("1. DATA CLEANING\n" + "-"*40 + "\n")
    f.write("\n".join(log_report) + "\n\n")
    f.write("2. PREPROCESSING\n" + "-"*40 + "\n")
    f.write("- Median imputation for missing A/G ratio values\n")
    f.write("- log1p transform applied to 5 right-skewed lab markers "
            "(bilirubin, ALP, ALT, AST) to reduce skewness from ~4-10 to ~1.2-1.7\n")
    f.write("- Gender encoded binary (Male=1, Female=0)\n")
    f.write("- StandardScaler applied after log transform\n")
    f.write("- All preprocessing fit INSIDE cross-validation folds only (no leakage)\n\n")
    f.write("3. MODEL COMPARISON (5-fold stratified CV, nested grid search "
            "for KNN/SVM/RF/GB)\n" + "-"*40 + "\n")
    f.write(results_df.to_string(index=False) + "\n\n")
    f.write("4. STATISTICAL SIGNIFICANCE TEST\n" + "-"*40 + "\n")
    f.write(sig_report + "\n\n")
    f.write("5. BEST MODEL\n" + "-"*40 + "\n")
    f.write(f"Selected: {best_name}\n")
    f.write(f"Best hyperparameters: {best_params}\n")
    f.write(f"Mean CV AUC: {ranked[0][1]:.3f}\n\n")
    f.write("6. CLASSIFICATION REPORT (out-of-fold predictions, best model)\n" + "-"*40 + "\n")
    f.write(class_report_str + "\n\n")
    f.write("7. TOP FEATURES (Random Forest importance)\n" + "-"*40 + "\n")
    f.write(importances.sort_values(ascending=False).round(3).to_string() + "\n")

print("\nAll outputs saved. Best model:", best_name)

              Model           auc           acc           rec          prec            f1           bal
  Gradient Boosting 0.716 ± 0.039 0.689 ± 0.033 0.887 ± 0.056 0.733 ± 0.014 0.802 ± 0.027 0.544 ± 0.029
K-Nearest Neighbors 0.729 ± 0.035 0.711 ± 0.037 0.892 ± 0.029 0.751 ± 0.032 0.815 ± 0.021 0.577 ± 0.061
Logistic Regression 0.743 ± 0.035 0.651 ± 0.037 0.596 ± 0.057 0.876 ± 0.032 0.707 ± 0.042 0.692 ± 0.033
        Naive Bayes 0.743 ± 0.029 0.635 ± 0.021 0.542 ± 0.043 0.911 ± 0.022 0.678 ± 0.029 0.704 ± 0.009
      Random Forest 0.728 ± 0.020 0.661 ± 0.025 0.623 ± 0.035 0.864 ± 0.015 0.723 ± 0.025 0.690 ± 0.021
   SVM (RBF Kernel) 0.752 ± 0.024 0.614 ± 0.040 0.515 ± 0.065 0.903 ± 0.024 0.653 ± 0.050 0.687 ± 0.026

Best model: SVM (RBF Kernel) (mean AUC=0.752)
2nd best  : Naive Bayes (mean AUC=0.743)
Paired t-test on per-fold AUC (SVM (RBF Kernel) vs Naive Bayes): t=0.521, p=0.630 (NOT statistically significant (p>=0.05))

Classification report for best model:
                preci

In [2]:
"""
SHAP Explainability for ILPD Unimodal Model (fixed version)
============================================================
RUN THIS ON KAGGLE (SHAP is pre-installed there).
Files needed: ILPD_cleaned.csv and unimodal_best_model.joblib
Update the paths below if your files are in /kaggle/input/...
"""
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt

# ---- 1. Load data + trained model ----
df = pd.read_csv('/kaggle/working/ILPD_cleaned.csv')
X = df.drop(columns='target')
y = df['target']

model = joblib.load('/kaggle/working/unimodal_best_model.joblib')  # saved SVM pipeline

# ---- 2. Build SHAP explainer (KernelExplainer, model-agnostic) ----
def pipeline_predict(x):
    if isinstance(x, np.ndarray):
        x = pd.DataFrame(x, columns=X.columns)
    return model.predict_proba(x)

background = shap.sample(X, 50, random_state=42)
explainer = shap.KernelExplainer(pipeline_predict, background)
shap_values = explainer.shap_values(X, nsamples=100)

# ---- 3. Select class 1 (Liver Patient), handling both SHAP output formats ----
if isinstance(shap_values, list):
    # old format: [class0 (n, f), class1 (n, f)]
    shap_values_patient = np.array(shap_values[1])
else:
    sv = np.array(shap_values)
    if sv.ndim == 3:
        # new format: (n_samples, n_features, n_classes)
        shap_values_patient = sv[:, :, 1]
    else:
        shap_values_patient = sv

base_value = (
    explainer.expected_value[1]
    if np.ndim(explainer.expected_value) > 0
    else explainer.expected_value
)

print("shap_values_patient shape:", shap_values_patient.shape)  # should be (570, n_features)

# ---- 4. Global summary plot ----
plt.figure()
shap.summary_plot(shap_values_patient, X, show=False)
plt.tight_layout()
plt.savefig('shap_summary_plot.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: shap_summary_plot.png")

# ---- 5. Individual patient explanation (force plot) ----
# Use positional index (not label) of the first "Liver Patient" row
patient_pos = int(np.where(y.values == 1)[0][0])

shap.force_plot(
    base_value,
    shap_values_patient[patient_pos, :],   # 1D: (n_features,)
    X.iloc[patient_pos, :],                # 1D row
    matplotlib=True,
    show=False,
    figsize=(20, 3),
    text_rotation=15
)
plt.savefig('shap_force_plot_single_patient.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: shap_force_plot_single_patient.png")

# ---- 5b. (Optional) Waterfall plot - cleaner alternative for papers ----
exp = shap.Explanation(
    values=shap_values_patient[patient_pos],
    base_values=base_value,
    data=X.iloc[patient_pos].values,
    feature_names=list(X.columns)
)
shap.plots.waterfall(exp, show=False)
plt.savefig('shap_waterfall_single_patient.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: shap_waterfall_single_patient.png")

# ---- 6. Save raw SHAP values for the report/paper ----
shap_df = pd.DataFrame(shap_values_patient, columns=X.columns)
shap_df.to_csv('shap_values_per_patient.csv', index=False)
mean_abs_shap = shap_df.abs().mean().sort_values(ascending=False)
print("\nMean |SHAP value| per feature (global importance):")
print(mean_abs_shap)
mean_abs_shap.to_csv('shap_mean_importance.csv')

print("\nDone. Use shap_summary_plot.png as your main SHAP figure in the paper.")

  0%|          | 0/570 [00:00<?, ?it/s]

shap_values_patient shape: (570, 10)
Saved: shap_summary_plot.png
Saved: shap_force_plot_single_patient.png
Saved: shap_waterfall_single_patient.png

Mean |SHAP value| per feature (global importance):
ast_sgot                0.036744
alkaline_phosphotase    0.034660
alt_sgpt                0.034332
direct_bilirubin        0.032515
total_bilirubin         0.030425
age                     0.023762
albumin                 0.022331
ag_ratio                0.020867
gender                  0.007057
total_proteins          0.002983
dtype: float64

Done. Use shap_summary_plot.png as your main SHAP figure in the paper.


In [3]:
import os
print(os.listdir('/kaggle/working/'))


['unimodal_roc_all_models.png', 'eda_class_age.png', 'unimodal_confusion_matrix.png', 'shap_mean_importance.csv', 'shap_values_per_patient.csv', 'unimodal_final_report.txt', 'unimodal_feature_importance.png', 'unimodal_best_model.joblib', 'ILPD_cleaned.csv', 'shap_summary_plot.png', 'unimodal_results_table.csv', 'eda_correlation.png', 'shap_force_plot_single_patient.png', 'eda_skewness.png', 'shap_waterfall_single_patient.png', '__notebook__.ipynb']


In [4]:
import zipfile
import os

# Path to working directory
work_dir = '/kaggle/working/'
zip_path = os.path.join(work_dir, 'output.zip')

# Create a new zip file
with zipfile.ZipFile(zip_path, 'w') as z:
    for file in os.listdir(work_dir):
        # Skip any existing zip files to avoid nesting
        if file.endswith('.zip'):
            continue
        z.write(os.path.join(work_dir, file), file)

print("Created:", zip_path)


Created: /kaggle/working/output.zip


In [5]:
"""
UNIMODAL v2 (ILPD) - Feature engineering + Stacking + Threshold tuning
=======================================================================
Kaggle pe seedha run karo. Sab kuch leakage-safe hai:
  - preprocessing pipeline ke andar (har fold ke train se hi seekhta hai)
  - threshold sirf TRAIN fold ke andar (inner CV) se choose hota hai
Test fold ko kabhi tune karne ke liye use nahi kiya gaya.
"""
import os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

from scipy import stats
from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score, accuracy_score,
                             matthews_corrcoef, recall_score, confusion_matrix)

RANDOM_STATE = 42
N_REPEATS = int(os.environ.get('N_REPEATS', 3))          # 5-fold x 3 repeats = 15 folds
DATA_PATH = os.environ.get(
    'ILPD_PATH',
    '/kaggle/input/datasets/sham1026/indian-liver-patient-dataset/Indian Liver Patient Dataset (ILPD).csv')
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

# ------------------------------------------------------------
# 1. LOAD + CLEAN (same as v1)
# ------------------------------------------------------------
df = pd.read_csv(DATA_PATH)
df.columns = ['age', 'gender', 'total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase',
              'alt_sgpt', 'ast_sgot', 'total_proteins', 'albumin', 'ag_ratio', 'target']
df = df.drop_duplicates().reset_index(drop=True)
df['gender'] = (df['gender'] == 'Male').astype(int)
df['target'] = (df['target'] == 1).astype(int)          # 1 = liver patient

# ------------------------------------------------------------
# 2. FEATURE ENGINEERING (row-wise, koi fitting nahi, to leakage ka risk nahi)
# ------------------------------------------------------------
def add_features(d):
    d = d.copy()
    d['de_ritis'] = d['ast_sgot'] / (d['alt_sgpt'] + 1)                  # AST/ALT ratio
    d['direct_ratio'] = d['direct_bilirubin'] / (d['total_bilirubin'] + 0.1)
    d['globulin'] = d['total_proteins'] - d['albumin']
    return d

X = add_features(df.drop(columns='target'))
y = df['target'].values

skew_cols = ['total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase',
             'alt_sgpt', 'ast_sgot', 'de_ritis']
num_cols = ['age', 'total_proteins', 'albumin', 'ag_ratio', 'globulin', 'direct_ratio']

def make_preprocessor():
    return ColumnTransformer([
        ('log', Pipeline([('imp', SimpleImputer(strategy='median')),
                          ('l', FunctionTransformer(np.log1p))]), skew_cols),
        ('num', SimpleImputer(strategy='median'), num_cols),
        ('bin', 'passthrough', ['gender']),
    ])

def make_pipeline(model):
    return Pipeline([('prep', make_preprocessor()), ('sc', StandardScaler()), ('m', model)])

# ------------------------------------------------------------
# 3. MODELS: single baselines + 2 ensembles
# ------------------------------------------------------------
def base_learners():
    return [
        ('svm', SVC(kernel='rbf', C=1, gamma='scale', probability=True,
                    class_weight='balanced', random_state=RANDOM_STATE)),
        ('rf', RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=5,
                                      class_weight='balanced', random_state=RANDOM_STATE)),
        ('lr', LogisticRegression(max_iter=3000, class_weight='balanced',
                                  random_state=RANDOM_STATE)),
    ]

models = {
    'SVM (RBF)': make_pipeline(base_learners()[0][1]),
    'Random Forest': make_pipeline(base_learners()[1][1]),
    'Logistic Reg': make_pipeline(base_learners()[2][1]),
    'Soft Voting (SVM+RF+LR)': make_pipeline(VotingClassifier(base_learners(), voting='soft')),
    'Stacking (SVM+RF+LR -> LR)': make_pipeline(StackingClassifier(
        estimators=base_learners(),
        final_estimator=LogisticRegression(class_weight='balanced', max_iter=3000),
        stack_method='predict_proba', cv=5)),
}

# ------------------------------------------------------------
# 4. HELPERS
# ------------------------------------------------------------
def best_threshold(y_true, p):
    """Inner OOF probabilities pe balanced accuracy maximize karne wala threshold."""
    grid = np.linspace(0.2, 0.8, 61)
    scores = [balanced_accuracy_score(y_true, (p >= t).astype(int)) for t in grid]
    return grid[int(np.argmax(scores))]

def metrics(y_true, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {'acc': accuracy_score(y_true, pred),
            'bal_acc': balanced_accuracy_score(y_true, pred),
            'mcc': matthews_corrcoef(y_true, pred),
            'sens': tp / (tp + fn) if (tp + fn) else np.nan,
            'spec': tn / (tn + fp) if (tn + fp) else np.nan}

# ------------------------------------------------------------
# 5. NESTED EVALUATION
# ------------------------------------------------------------
outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
inner = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE + 1)

fold_rows = []
for k, (tr, te) in enumerate(outer.split(X, y)):
    Xtr, Xte, ytr, yte = X.iloc[tr], X.iloc[te], y[tr], y[te]
    for name, mdl in models.items():
        # threshold: sirf train fold ke andar se
        inner_p = cross_val_predict(clone(mdl), Xtr, ytr, cv=inner, method='predict_proba')[:, 1]
        thr = best_threshold(ytr, inner_p)
        m = clone(mdl).fit(Xtr, ytr)
        p = m.predict_proba(Xte)[:, 1]
        row = {'fold': k, 'model': name, 'thr': thr, 'auc': roc_auc_score(yte, p)}
        for tag, t in (('def', 0.5), ('tuned', thr)):
            for mk, mv in metrics(yte, p, t).items():
                row[f'{mk}_{tag}'] = mv
        fold_rows.append(row)
    print(f'fold {k + 1}/{5 * N_REPEATS} done')

res = pd.DataFrame(fold_rows)

# ------------------------------------------------------------
# 6. SUMMARY TABLE
# ------------------------------------------------------------
cols = ['auc', 'bal_acc_def', 'bal_acc_tuned', 'acc_tuned', 'mcc_tuned', 'sens_tuned', 'spec_tuned']
summary = res.groupby('model')[cols].agg(['mean', 'std'])
table = pd.DataFrame({c: summary[c]['mean'].round(3).astype(str) + ' ± ' +
                         summary[c]['std'].round(3).astype(str) for c in cols})
print('\n' + '=' * 90)
print('Majority-class baseline accuracy (sab ko "patient" bol do): '
      f'{max(y.mean(), 1 - y.mean()):.3f}')
print('=' * 90)
print(table.to_string())
table.to_csv(os.path.join(OUT_DIR, 'unimodal_v2_results.csv'))

# ------------------------------------------------------------
# 7. SIGNIFICANCE: Nadeau-Bengio corrected paired t-test
#    (repeated CV ke folds independent nahi hote, isliye normal t-test over-optimistic hota hai)
# ------------------------------------------------------------
def corrected_ttest(a, b, n_train, n_test):
    d = np.asarray(a) - np.asarray(b)
    k = len(d)
    var = np.var(d, ddof=1) * (1 / k + n_test / n_train)
    t = d.mean() / np.sqrt(var) if var > 0 else 0.0
    return d.mean(), t, 2 * stats.t.sf(abs(t), df=k - 1)

n_test = len(y) // 5
n_train = len(y) - n_test
piv_auc = res.pivot(index='fold', columns='model', values='auc')
piv_bal = res.pivot(index='fold', columns='model', values='bal_acc_tuned')
print('\nCorrected paired t-test (Stacking vs SVM baseline):')
for label, piv in (('AUC', piv_auc), ('Balanced acc (tuned thr)', piv_bal)):
    diff, t, p = corrected_ttest(piv['Stacking (SVM+RF+LR -> LR)'], piv['SVM (RBF)'], n_train, n_test)
    print(f'  {label:26s} mean diff {diff:+.3f} | t={t:.2f} | p={p:.3f}')

# ------------------------------------------------------------
# 8. FINAL MODEL (best by mean AUC), full data pe refit + threshold save
# ------------------------------------------------------------
import joblib
best = res.groupby('model')['auc'].mean().idxmax()
final = clone(models[best])
oof = cross_val_predict(clone(final), X, y, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                        method='predict_proba')[:, 1]
final_thr = best_threshold(y, oof)
final.fit(X, y)
joblib.dump({'model': final, 'threshold': final_thr}, os.path.join(OUT_DIR, 'unimodal_v2_best.joblib'))
print(f'\nBest by AUC: {best} | saved with threshold {final_thr:.2f}')

fold 1/15 done
fold 2/15 done
fold 3/15 done
fold 4/15 done
fold 5/15 done
fold 6/15 done
fold 7/15 done
fold 8/15 done
fold 9/15 done
fold 10/15 done
fold 11/15 done
fold 12/15 done
fold 13/15 done
fold 14/15 done
fold 15/15 done

Majority-class baseline accuracy (sab ko "patient" bol do): 0.712
                                      auc    bal_acc_def  bal_acc_tuned      acc_tuned      mcc_tuned     sens_tuned     spec_tuned
model                                                                                                                              
Logistic Reg                 0.75 ± 0.026   0.69 ± 0.023  0.684 ± 0.019   0.62 ± 0.046  0.344 ± 0.036  0.533 ± 0.096  0.836 ± 0.092
Random Forest               0.734 ± 0.032  0.676 ± 0.038  0.692 ± 0.027  0.629 ± 0.037  0.353 ± 0.048  0.544 ± 0.066   0.839 ± 0.07
SVM (RBF)                   0.745 ± 0.032  0.566 ± 0.034  0.684 ± 0.027  0.625 ± 0.034   0.337 ± 0.05  0.544 ± 0.057  0.823 ± 0.063
Soft Voting (SVM+RF+LR)     0.748 ± 0.028 

In [6]:
"""
ILPD DATA AUDIT (sirf check karta hai, data ko badalta nahi)
============================================================
Pehle dekho ki data mein asal mein kya gandagi hai, phir decide karo ki kya clean karna hai.
Kaggle pe seedha run karo.
"""
import os
import numpy as np, pandas as pd

DATA_PATH = os.environ.get(
    'ILPD_PATH',
    '/kaggle/input/datasets/sham1026/indian-liver-patient-dataset/Indian Liver Patient Dataset (ILPD).csv')

df = pd.read_csv(DATA_PATH)
df.columns = ['age', 'gender', 'total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase',
              'alt_sgpt', 'ast_sgot', 'total_proteins', 'albumin', 'ag_ratio', 'target']
feat = [c for c in df.columns if c != 'target']
print(f'Raw shape: {df.shape}\n')

# 1. DUPLICATES
exact = df.duplicated().sum()
dup_feat = df[df.duplicated(subset=feat, keep=False)]
conflict = dup_feat.groupby(feat)['target'].nunique()
print('1. DUPLICATES')
print(f'   exact duplicate rows            : {exact}')
print(f'   same features, DIFFERENT label  : {(conflict > 1).sum()}  (label conflict = noise)\n')

df = df.drop_duplicates().reset_index(drop=True)

# 2. MISSING
print('2. MISSING VALUES')
miss = df.isna().sum()
print(miss[miss > 0].to_string() if miss.any() else '   koi nahi')
print()

# 3. LOGICAL ERRORS (medical rules)
print('3. LOGICAL / DOMAIN CHECKS')
bad_bili = df['direct_bilirubin'] > df['total_bilirubin']
bad_prot = df['albumin'] > df['total_proteins']
print(f'   direct_bilirubin > total_bilirubin : {bad_bili.sum()}  (impossible)')
print(f'   albumin > total_proteins           : {bad_prot.sum()}  (impossible)')

globulin = df['total_proteins'] - df['albumin']
ag_calc = df['albumin'] / globulin.replace(0, np.nan)
both = df['ag_ratio'].notna() & ag_calc.notna()
if both.sum() > 2:
    corr = np.corrcoef(df.loc[both, 'ag_ratio'], ag_calc[both])[0, 1]
    mae = (df.loc[both, 'ag_ratio'] - ag_calc[both]).abs().median()
    print(f'   ag_ratio vs albumin/globulin       : corr={corr:.3f}, median abs diff={mae:.3f}')
    if corr > 0.9:
        print('   -> ag_ratio formula se nikal sakte ho. Missing ag_ratio ko median ki jagah')
        print('      albumin/(total_proteins-albumin) se bharna zyada sahi imputation hai.')
print(f'   age <= 0 ya > 100                  : {((df.age <= 0) | (df.age > 100)).sum()}')
print(f'   gender values                      : {df.gender.unique().tolist()}')
print(f'   target values                      : {df.target.unique().tolist()}\n')

# 4. OUTLIERS (log scale pe IQR, kyunki lab values right-skewed hain)
print('4. EXTREME VALUES (log1p scale, IQR rule x3)')
skew_cols = ['total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase', 'alt_sgpt', 'ast_sgot']
for c in skew_cols:
    v = np.log1p(df[c])
    q1, q3 = v.quantile([0.25, 0.75])
    lo, hi = q1 - 3 * (q3 - q1), q3 + 3 * (q3 - q1)
    n_out = ((v < lo) | (v > hi)).sum()
    print(f'   {c:22s} max={df[c].max():8.1f}  p99={df[c].quantile(.99):8.1f}  extreme rows={n_out}')
print('   NOTE: liver disease mein bahut high values REAL hote hain (yahi signal hai).')
print('   Inhe delete mat karo; zarurat ho to winsorize (clip) karo.\n')

# 5. CLASS / GROUP BALANCE
print('5. BALANCE')
print(df['target'].value_counts().rename({1: 'patient', 2: 'healthy', 0: 'healthy'}).to_string())
df['is_patient'] = (df['target'] == 1).astype(int)
print(df.groupby('gender')['is_patient'].agg(['count', 'mean']).round(3).rename(columns={'mean': 'patient_rate'}).to_string())
print('\nAudit complete. Ab batao kaun si cheez sach mein gandi nikli.')

Raw shape: (583, 11)

1. DUPLICATES
   exact duplicate rows            : 13
   same features, DIFFERENT label  : 0  (label conflict = noise)

2. MISSING VALUES
ag_ratio    4

3. LOGICAL / DOMAIN CHECKS
   direct_bilirubin > total_bilirubin : 3  (impossible)
   albumin > total_proteins           : 0  (impossible)
   ag_ratio vs albumin/globulin       : corr=0.847, median abs diff=0.031
   age <= 0 ya > 100                  : 0
   gender values                      : ['Female', 'Male']
   target values                      : [1, 2]

4. EXTREME VALUES (log1p scale, IQR rule x3)
   total_bilirubin        max=    75.0  p99=    28.6  extreme rows=6
   direct_bilirubin       max=    19.7  p99=    13.1  extreme rows=3
   alkaline_phosphotase   max=  2110.0  p99=  1559.3  extreme rows=7
   alt_sgpt               max=  2000.0  p99=  1043.0  extreme rows=6
   ast_sgot               max=  4929.0  p99=   987.9  extreme rows=1
   NOTE: liver disease mein bahut high values REAL hote hain (yahi signal

In [7]:
"""
UNIMODAL v3 (ILPD) - clean data + feature engineering + many algorithms + stacking
===================================================================================
Kaggle pe seedha run karo. Leakage-safe:
  - duplicate/impossible rows split se PEHLE hatate hain (row-level, kuch seekhna nahi padta)
  - imputation / log / scaling pipeline ke andar (har fold ke train se hi seekhta hai)
  - threshold sirf TRAIN fold ke andar (inner CV) se choose hota hai
Outputs (/kaggle/working):
  ILPD_cleaned.csv          -> engineered features ke saath (SHAP script isi ko padhegi)
  unimodal_best_model.joblib-> sirf pipeline (SHAP script bina badle chalegi)
  unimodal_best_threshold.txt
  unimodal_v3_results.csv
"""
import os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

from scipy import stats
from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              HistGradientBoostingClassifier, StackingClassifier, VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score, accuracy_score,
                             matthews_corrcoef, confusion_matrix)
import joblib

RANDOM_STATE = 42
N_REPEATS = int(os.environ.get('N_REPEATS', 3))        # 5-fold x 3 = 15 outer folds
DATA_PATH = os.environ.get(
    'ILPD_PATH',
    '/kaggle/input/datasets/sham1026/indian-liver-patient-dataset/Indian Liver Patient Dataset (ILPD).csv')
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

# ------------------------------------------------------------
# 1. LOAD + CLEAN
# ------------------------------------------------------------
df = pd.read_csv(DATA_PATH)
df.columns = ['age', 'gender', 'total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase',
              'alt_sgpt', 'ast_sgot', 'total_proteins', 'albumin', 'ag_ratio', 'target']
n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
n_dup = n0 - len(df)

bad = df['direct_bilirubin'] > df['total_bilirubin']          # medically impossible
n_bad = int(bad.sum())
df = df[~bad].reset_index(drop=True)

df['gender'] = (df['gender'] == 'Male').astype(int)
df['target'] = (df['target'] == 1).astype(int)                # 1 = liver patient
print(f'Raw {n0} -> removed {n_dup} duplicates, {n_bad} impossible-bilirubin rows -> {len(df)} rows')

# ------------------------------------------------------------
# 2. FEATURE ENGINEERING (row-wise, koi fitting nahi => leakage nahi)
# ------------------------------------------------------------
df['de_ritis'] = df['ast_sgot'] / (df['alt_sgpt'] + 1)                    # AST/ALT
df['direct_ratio'] = df['direct_bilirubin'] / (df['total_bilirubin'] + 0.1)
df['globulin'] = df['total_proteins'] - df['albumin']
df.to_csv(os.path.join(OUT_DIR, 'ILPD_cleaned.csv'), index=False)

X = df.drop(columns='target')
y = df['target'].values
base_rate = max(y.mean(), 1 - y.mean())
print(f'Majority-class baseline accuracy: {base_rate:.3f}  (isse upar jaana hi asli kamyabi hai)')

skew_cols = ['total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase',
             'alt_sgpt', 'ast_sgot', 'de_ritis']
num_cols = ['age', 'total_proteins', 'albumin', 'ag_ratio', 'globulin', 'direct_ratio']

def make_pipeline(model):
    prep = ColumnTransformer([
        ('log', Pipeline([('imp', SimpleImputer(strategy='median')),
                          ('l', FunctionTransformer(np.log1p))]), skew_cols),
        ('num', SimpleImputer(strategy='median'), num_cols),
        ('bin', 'passthrough', ['gender']),
    ])
    return Pipeline([('prep', prep), ('sc', StandardScaler()), ('m', model)])

# ------------------------------------------------------------
# 3. ALGORITHMS (chhota data hai => regularized / shallow settings)
# ------------------------------------------------------------
def learners():
    d = {
        'LR': LogisticRegression(C=0.5, max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE),
        'SVM': SVC(kernel='rbf', C=1, gamma='scale', probability=True,
                   class_weight='balanced', random_state=RANDOM_STATE),
        'RF': RandomForestClassifier(n_estimators=500, max_depth=6, min_samples_leaf=5,
                                     class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1),
        'ExtraTrees': ExtraTreesClassifier(n_estimators=600, min_samples_leaf=4, max_features='sqrt',
                                           class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1),
        'HistGB': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.03, max_iter=250,
                                                 l2_regularization=2.0, min_samples_leaf=15,
                                                 class_weight='balanced', random_state=RANDOM_STATE),
    }
    try:
        from xgboost import XGBClassifier
        d['XGBoost'] = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8,
                                     colsample_bytree=0.8, min_child_weight=3, reg_lambda=3.0,
                                     eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)
    except Exception:
        print('xgboost nahi mila, skip')
    try:
        from lightgbm import LGBMClassifier
        d['LightGBM'] = LGBMClassifier(n_estimators=300, max_depth=3, num_leaves=7, learning_rate=0.03,
                                       subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                       min_child_samples=15, reg_lambda=3.0, class_weight='balanced',
                                       random_state=RANDOM_STATE, verbose=-1, n_jobs=-1)
    except Exception:
        print('lightgbm nahi mila, skip')
    try:
        from catboost import CatBoostClassifier
        d['CatBoost'] = CatBoostClassifier(iterations=400, depth=4, learning_rate=0.03, l2_leaf_reg=5,
                                           auto_class_weights='Balanced', random_seed=RANDOM_STATE,
                                           verbose=0, thread_count=-1)
    except Exception:
        print('catboost nahi mila, skip')
    return d

L = learners()
ens_members = [(k, L[k]) for k in ['LR', 'SVM', 'ExtraTrees', 'HistGB'] if k in L]
if 'CatBoost' in L:
    ens_members.append(('CatBoost', L['CatBoost']))

models = {name: make_pipeline(m) for name, m in L.items()}
models['Soft Voting'] = make_pipeline(VotingClassifier(
    [(k, clone(v)) for k, v in ens_members], voting='soft'))
models['Stacking -> LR'] = make_pipeline(StackingClassifier(
    estimators=[(k, clone(v)) for k, v in ens_members],
    final_estimator=LogisticRegression(C=0.5, class_weight='balanced', max_iter=3000),
    stack_method='predict_proba', cv=5))

# ------------------------------------------------------------
# 4. HELPERS
# ------------------------------------------------------------
GRID = np.linspace(0.2, 0.8, 61)

def best_threshold(y_true, p, kind='bal'):
    f = balanced_accuracy_score if kind == 'bal' else accuracy_score
    return GRID[int(np.argmax([f(y_true, (p >= t).astype(int)) for t in GRID]))]

def metrics(y_true, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {'acc': accuracy_score(y_true, pred),
            'bal_acc': balanced_accuracy_score(y_true, pred),
            'mcc': matthews_corrcoef(y_true, pred),
            'sens': tp / (tp + fn) if (tp + fn) else np.nan,
            'spec': tn / (tn + fp) if (tn + fp) else np.nan}

# ------------------------------------------------------------
# 5. NESTED EVALUATION (outer = honest test, inner = threshold choose)
# ------------------------------------------------------------
outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
inner = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE + 1)

rows = []
total = 5 * N_REPEATS
for k, (tr, te) in enumerate(outer.split(X, y)):
    Xtr, Xte, ytr, yte = X.iloc[tr], X.iloc[te], y[tr], y[te]
    for name, mdl in models.items():
        inner_p = cross_val_predict(clone(mdl), Xtr, ytr, cv=inner, method='predict_proba')[:, 1]
        thr_bal = best_threshold(ytr, inner_p, 'bal')
        thr_acc = best_threshold(ytr, inner_p, 'acc')
        p = clone(mdl).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
        row = {'fold': k, 'model': name, 'auc': roc_auc_score(yte, p)}
        for tag, t in (('def', 0.5), ('bal', thr_bal), ('acc', thr_acc)):
            for mk, mv in metrics(yte, p, t).items():
                row[f'{mk}_{tag}'] = mv
        rows.append(row)
    print(f'fold {k + 1}/{total} done', flush=True)

res = pd.DataFrame(rows)

# ------------------------------------------------------------
# 6. SUMMARY
#   *_def  = threshold 0.5
#   *_bal  = threshold balanced-accuracy ke liye tuned (sensitivity/specificity balanced)
#   *_acc  = threshold accuracy ke liye tuned (accuracy zyada, par specificity gir sakti hai)
# ------------------------------------------------------------
cols = ['auc', 'acc_def', 'acc_acc', 'bal_acc_bal', 'mcc_bal', 'sens_bal', 'spec_bal']
g = res.groupby('model')[cols]
table = pd.DataFrame({c: g.mean()[c].round(3).astype(str) + ' ± ' + g.std()[c].round(3).astype(str)
                      for c in cols})
table = table.loc[g.mean().sort_values('auc', ascending=False).index]
print('\n' + '=' * 100)
print(f'Majority baseline accuracy = {base_rate:.3f}')
print('=' * 100)
print(table.to_string())
table.to_csv(os.path.join(OUT_DIR, 'unimodal_v3_results.csv'))

# ------------------------------------------------------------
# 7. SIGNIFICANCE: Nadeau-Bengio corrected paired t-test (best vs Logistic Regression)
# ------------------------------------------------------------
def corrected_ttest(a, b, n_train, n_test):
    d = np.asarray(a) - np.asarray(b)
    kk = len(d)
    var = np.var(d, ddof=1) * (1 / kk + n_test / n_train)
    t = d.mean() / np.sqrt(var) if var > 0 else 0.0
    return d.mean(), t, 2 * stats.t.sf(abs(t), df=kk - 1)

best = g.mean()['auc'].idxmax()
n_test = len(y) // 5
n_train = len(y) - n_test
piv = res.pivot(index='fold', columns='model', values='auc')
diff, t, p = corrected_ttest(piv[best], piv['LR'], n_train, n_test)
print(f'\nAUC: {best} vs LR -> mean diff {diff:+.3f} | t={t:.2f} | p={p:.3f}')

# ------------------------------------------------------------
# 8. FINAL MODEL: best by mean AUC, full data pe refit
#    (note: selection aur reporting same CV pe hui hai => thoda optimistic, paper mein likho)
# ------------------------------------------------------------
final = clone(models[best])
oof = cross_val_predict(clone(final), X, y, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                        method='predict_proba')[:, 1]
final_thr = best_threshold(y, oof, 'bal')
final.fit(X, y)
joblib.dump(final, os.path.join(OUT_DIR, 'unimodal_best_model.joblib'))
with open(os.path.join(OUT_DIR, 'unimodal_best_threshold.txt'), 'w') as f:
    f.write(f'{best}\nthreshold={final_thr:.2f}\n')
print(f'\nBest by AUC: {best} | threshold {final_thr:.2f} | saved.')

Raw 583 -> removed 13 duplicates, 3 impossible-bilirubin rows -> 567 rows
Majority-class baseline accuracy: 0.711  (isse upar jaana hi asli kamyabi hai)
fold 1/15 done
fold 2/15 done
fold 3/15 done
fold 4/15 done
fold 5/15 done
fold 6/15 done
fold 7/15 done
fold 8/15 done
fold 9/15 done
fold 10/15 done
fold 11/15 done
fold 12/15 done
fold 13/15 done
fold 14/15 done
fold 15/15 done

Majority baseline accuracy = 0.711
                          auc        acc_def        acc_acc    bal_acc_bal        mcc_bal       sens_bal       spec_bal
model                                                                                                                  
LR               0.75 ± 0.036  0.651 ± 0.038  0.718 ± 0.026  0.676 ± 0.031  0.328 ± 0.059  0.541 ± 0.104  0.811 ± 0.105
Stacking -> LR  0.744 ± 0.034  0.651 ± 0.038  0.717 ± 0.013  0.685 ± 0.036   0.34 ± 0.065  0.547 ± 0.072  0.823 ± 0.064
ExtraTrees      0.744 ± 0.039  0.661 ± 0.035   0.71 ± 0.027  0.691 ± 0.044   0.353 ± 0.08  0.543 ± 0

In [8]:
"""
UNIMODAL v4 - Feature ablation + L1 selection + tuned regularization (ILPD)
===========================================================================
Pehle unimodal_v3.py run karo (wo ILPD_cleaned.csv banata hai), phir ye.

Leakage-safe design:
  - Har experiment ke liye SAME outer CV splits (5-fold x 3 repeats) => paired comparison
  - Feature selection / C tuning pipeline ya GridSearchCV ke ANDAR hai => test fold kabhi nahi dikhta
  - Har config ko FULL baseline se Nadeau-Bengio corrected t-test se compare kiya jaata hai
Sawal: kya koi feature hatane / L1 / tuning se AUC SIGNIFICANTLY badhta hai?
"""
import os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

from scipy import stats
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel

RANDOM_STATE = 42
N_REPEATS = int(os.environ.get('N_REPEATS', 3))
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
CSV = os.path.join(OUT_DIR, 'ILPD_cleaned.csv')

df = pd.read_csv(CSV)
X = df.drop(columns='target')
y = df['target'].values
print(f'Loaded {CSV}: {X.shape[0]} rows, {X.shape[1]} features')

SKEW = ['total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase', 'alt_sgpt', 'ast_sgot', 'de_ritis']
NUM = ['age', 'total_proteins', 'albumin', 'ag_ratio', 'globulin', 'direct_ratio']
BIN = ['gender']
ALL = SKEW + NUM + BIN                                  # 13 features
ORIGINAL = ['age', 'gender', 'total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase',
            'alt_sgpt', 'ast_sgot', 'total_proteins', 'albumin', 'ag_ratio']
ENGINEERED = ['de_ritis', 'direct_ratio', 'globulin']

# ------------------------------------------------------------
# Pipeline builder: kisi bhi feature subset ke liye
# (ColumnTransformer sirf wahi columns leta hai jo cols mein hon, baaki drop)
# ------------------------------------------------------------
def build(cols, model):
    t = []
    lg = [c for c in SKEW if c in cols]
    nm = [c for c in NUM if c in cols]
    bn = [c for c in BIN if c in cols]
    if lg:
        t.append(('log', Pipeline([('imp', SimpleImputer(strategy='median')),
                                   ('l', FunctionTransformer(np.log1p))]), lg))
    if nm:
        t.append(('num', SimpleImputer(strategy='median'), nm))
    if bn:
        t.append(('bin', 'passthrough', bn))
    return Pipeline([('prep', ColumnTransformer(t)), ('sc', StandardScaler()), ('m', model)])

def lr_l2(C=0.5):
    return LogisticRegression(C=C, max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE)

def lr_l1(C=0.1):
    return LogisticRegression(penalty='l1', solver='liblinear', C=C, max_iter=3000,
                              class_weight='balanced', random_state=RANDOM_STATE)

outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
inner = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE + 1)
scoring = {'auc': 'roc_auc', 'bal': 'balanced_accuracy', 'acc': 'accuracy'}

n_test = len(y) // 5
n_train = len(y) - n_test

def corrected_ttest(a, b):
    d = np.asarray(a) - np.asarray(b)
    k = len(d)
    var = np.var(d, ddof=1) * (1 / k + n_test / n_train)
    if var <= 0:
        return d.mean(), 0.0, 1.0
    t = d.mean() / np.sqrt(var)
    return d.mean(), t, 2 * stats.t.sf(abs(t), df=k - 1)

# ------------------------------------------------------------
# Experiments
# ------------------------------------------------------------
experiments = {}   # name -> (estimator, n_features)

experiments['BASELINE: LR (L2, C=0.5), all 13'] = (build(ALL, lr_l2()), 13)
experiments['Original 10 only (no engineered)'] = (build(ORIGINAL, lr_l2()), 10)

# leave-one-feature-out
for f in ALL:
    cols = [c for c in ALL if c != f]
    experiments[f'drop {f}'] = (build(cols, lr_l2()), 12)

# correlated groups hatana
groups = {
    'drop bilirubin block (TB, DB, direct_ratio)': ['total_bilirubin', 'direct_bilirubin', 'direct_ratio'],
    'drop enzyme block (ALT, AST, de_ritis)': ['alt_sgpt', 'ast_sgot', 'de_ritis'],
    'drop protein block (TP, Alb, AG, globulin)': ['total_proteins', 'albumin', 'ag_ratio', 'globulin'],
    'drop age + gender': ['age', 'gender'],
}
for gname, gcols in groups.items():
    cols = [c for c in ALL if c not in gcols]
    experiments[gname] = (build(cols, lr_l2()), len(cols))

# tuned C (nested: GridSearchCV outer CV ke train fold ke andar chalta hai)
Cs = [0.01, 0.05, 0.1, 0.25, 0.5, 1, 2, 5]
experiments['LR L2, C tuned (nested)'] = (
    GridSearchCV(build(ALL, lr_l2()), {'m__C': Cs}, scoring='roc_auc', cv=inner), 13)
experiments['LR L1, C tuned (nested)'] = (
    GridSearchCV(build(ALL, lr_l1()), {'m__C': Cs}, scoring='roc_auc', cv=inner), 13)

# L1 se feature select -> phir L2 LR (selection har fold ke train mein)
sel_pipe = Pipeline([
    ('prep', build(ALL, lr_l2()).named_steps['prep']),
    ('sc', StandardScaler()),
    ('sel', SelectFromModel(lr_l1(C=0.1))),
    ('m', lr_l2()),
])
experiments['L1-select (C=0.1) -> L2 LR'] = (sel_pipe, 'auto')

# ------------------------------------------------------------
# Run
# ------------------------------------------------------------
rows, fold_auc = [], {}
for name, (est, nf) in experiments.items():
    r = cross_validate(est, X, y, cv=outer, scoring=scoring, n_jobs=-1)
    fold_auc[name] = r['test_auc']
    rows.append({'config': name, 'n_feat': nf,
                 'auc_mean': r['test_auc'].mean(), 'auc_std': r['test_auc'].std(),
                 'bal_acc': r['test_bal'].mean(), 'acc': r['test_acc'].mean()})
    print(f'done: {name}', flush=True)

base_name = 'BASELINE: LR (L2, C=0.5), all 13'
base = fold_auc[base_name]
for row in rows:
    diff, t, p = corrected_ttest(fold_auc[row['config']], base)
    row['dAUC_vs_base'], row['p_vs_base'] = diff, p

res = pd.DataFrame(rows).sort_values('auc_mean', ascending=False).reset_index(drop=True)
show = res.copy()
show['AUC'] = show['auc_mean'].round(3).astype(str) + ' ± ' + show['auc_std'].round(3).astype(str)
show['bal_acc'] = show['bal_acc'].round(3)
show['acc'] = show['acc'].round(3)
show['dAUC_vs_base'] = show['dAUC_vs_base'].map(lambda v: f'{v:+.3f}')
show['p_vs_base'] = show['p_vs_base'].round(3)
print('\n' + '=' * 110)
print(show[['config', 'n_feat', 'AUC', 'bal_acc', 'acc', 'dAUC_vs_base', 'p_vs_base']].to_string(index=False))
res.to_csv(os.path.join(OUT_DIR, 'unimodal_v4_ablation.csv'), index=False)

print('\nPadhne ka tareeka:')
print(' - p_vs_base < 0.05 aur dAUC > +0.01 ho tabhi "improvement" bolo.')
print(' - ~20 comparisons ho rahe hain, to 1-2 random p<0.05 sirf chance se aa sakte hain (multiple testing).')
print(' - dAUC ~ 0 aur n_feat kam => "kam features mein same performance" (simpler model) bhi ek valid finding hai.')

# ------------------------------------------------------------
# Descriptive: full data pe L1 kya zero karta hai (SIRF dekhne ke liye, performance claim nahi)
# ------------------------------------------------------------
names = SKEW + NUM + BIN
print('\nL1 coefficients (full-data refit, descriptive only):')
coef_tab = {}
for C in [0.05, 0.1, 0.25, 0.5]:
    m = build(ALL, lr_l1(C)).fit(X, y)
    coef_tab[f'C={C}'] = m.named_steps['m'].coef_.ravel()
coef_df = pd.DataFrame(coef_tab, index=names).round(3)
print(coef_df.to_string())
coef_df.to_csv(os.path.join(OUT_DIR, 'unimodal_v4_l1_coefficients.csv'))
print('\nSaved: unimodal_v4_ablation.csv, unimodal_v4_l1_coefficients.csv')

Loaded /kaggle/working/ILPD_cleaned.csv: 567 rows, 13 features
done: BASELINE: LR (L2, C=0.5), all 13
done: Original 10 only (no engineered)
done: drop total_bilirubin
done: drop direct_bilirubin
done: drop alkaline_phosphotase
done: drop alt_sgpt
done: drop ast_sgot
done: drop de_ritis
done: drop age
done: drop total_proteins
done: drop albumin
done: drop ag_ratio
done: drop globulin
done: drop direct_ratio
done: drop gender
done: drop bilirubin block (TB, DB, direct_ratio)
done: drop enzyme block (ALT, AST, de_ritis)
done: drop protein block (TP, Alb, AG, globulin)
done: drop age + gender
done: LR L2, C tuned (nested)
done: LR L1, C tuned (nested)
done: L1-select (C=0.1) -> L2 LR

                                     config n_feat           AUC  bal_acc   acc dAUC_vs_base  p_vs_base
                                drop gender     12 0.754 ± 0.034    0.696 0.657       +0.004      0.307
                 L1-select (C=0.1) -> L2 LR   auto 0.752 ± 0.034    0.700 0.659       +0.002      0.

In [9]:
"""
SHAP Explainability (LinearExplainer) for ILPD unimodal Logistic Regression
===========================================================================
Pehle unimodal_v3.py run karo. Ye files chahiye (/kaggle/working mein):
  ILPD_cleaned.csv, unimodal_best_model.joblib   (v3 ka best model = LR pipeline)

LinearExplainer exact hai aur seconds mein chalta hai (KernelExplainer ki zarurat nahi).
SHAP values log-odds (decision function) ki unit mein hote hain, probability mein nahi.
"""
import os
import numpy as np, pandas as pd
import joblib, shap
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

# ---- 1. Load ----
df = pd.read_csv(os.path.join(OUT_DIR, 'ILPD_cleaned.csv'))
X = df.drop(columns='target')
y = df['target'].values
model = joblib.load(os.path.join(OUT_DIR, 'unimodal_best_model.joblib'))

lr = model.named_steps['m']
if not isinstance(lr, LogisticRegression):
    raise SystemExit(f'Saved model {type(lr).__name__} hai, LR nahi. '
                     'Is script ke liye v3 mein LR best hona chahiye (ya purani KernelExplainer script use karo).')

# ---- 2. Preprocessing ke baad ka data (model ka asli input) ----
# Feature order v3 ke ColumnTransformer jaisa: log cols, num cols, gender
SKEW = ['total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase', 'alt_sgpt', 'ast_sgot', 'de_ritis']
NUM = ['age', 'total_proteins', 'albumin', 'ag_ratio', 'globulin', 'direct_ratio']
names = SKEW + NUM + ['gender']

Xt = model[:-1].transform(X)                    # imputer + log1p + scaler
assert Xt.shape[1] == len(names), f'{Xt.shape[1]} vs {len(names)} features'
X_disp = X[names]                               # plots mein asli values dikhane ke liye

# ---- 3. SHAP ----
explainer = shap.LinearExplainer(lr, Xt)
sv = np.asarray(explainer.shap_values(Xt))
if sv.ndim == 3:                                # kuch versions mein extra axis aata hai
    sv = sv[:, :, 1]
base = float(np.ravel(explainer.expected_value)[-1])
print('SHAP shape:', sv.shape)

# sanity check: base + sum(SHAP) == model ka log-odds
err = np.abs(sv.sum(axis=1) + base - lr.decision_function(Xt)).max()
print(f'Additivity check (max error vs decision_function): {err:.2e}')

# ---- 4. Global plots ----
plt.figure()
shap.summary_plot(sv, X_disp, show=False)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'shap_summary_plot.png'), dpi=150, bbox_inches='tight')
plt.close()

plt.figure()
shap.summary_plot(sv, X_disp, plot_type='bar', show=False)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'shap_bar_plot.png'), dpi=150, bbox_inches='tight')
plt.close()
print('Saved: shap_summary_plot.png, shap_bar_plot.png')

# ---- 5. Ek patient ka explanation ----
pos = int(np.where(y == 1)[0][0])
exp = shap.Explanation(values=sv[pos], base_values=base,
                       data=X_disp.iloc[pos].values, feature_names=names)
plt.figure()
shap.plots.waterfall(exp, show=False)
plt.savefig(os.path.join(OUT_DIR, 'shap_waterfall_single_patient.png'), dpi=150, bbox_inches='tight')
plt.close()

try:
    shap.force_plot(base, sv[pos], X_disp.iloc[pos], matplotlib=True, show=False,
                    figsize=(20, 3), text_rotation=15)
    plt.savefig(os.path.join(OUT_DIR, 'shap_force_plot_single_patient.png'), dpi=150, bbox_inches='tight')
    plt.close()
except Exception as e:
    print('force plot skip:', e)
print('Saved: patient waterfall/force plots')

# ---- 6. Tables ----
shap_df = pd.DataFrame(sv, columns=names)
shap_df.to_csv(os.path.join(OUT_DIR, 'shap_values_per_patient.csv'), index=False)

imp = shap_df.abs().mean().sort_values(ascending=False)
coef = pd.Series(lr.coef_.ravel(), index=names)
cmp_tab = pd.DataFrame({'mean_abs_shap': imp.round(3), 'LR_coef(std. scale)': coef[imp.index].round(3)})
print('\nGlobal importance (mean |SHAP|) vs LR coefficient:')
print(cmp_tab.to_string())
cmp_tab.to_csv(os.path.join(OUT_DIR, 'shap_mean_importance.csv'))

print('\nNote: bilirubin/enzyme/protein features aapas mein correlated hain, SHAP unke beech credit baant deta hai.')
print('Isliye individual feature ko "sabse zaruri" kehne ki jagah feature GROUPS ki baat karo.')

SHAP shape: (567, 13)
Additivity check (max error vs decision_function): 1.78e-15
Saved: shap_summary_plot.png, shap_bar_plot.png
Saved: patient waterfall/force plots

Global importance (mean |SHAP|) vs LR coefficient:
                      mean_abs_shap  LR_coef(std. scale)
direct_bilirubin              0.367                0.504
globulin                      0.353                0.463
albumin                       0.338               -0.414
ast_sgot                      0.297                0.391
alt_sgpt                      0.285                0.390
age                           0.237                0.288
ag_ratio                      0.234                0.313
total_bilirubin               0.219                0.301
alkaline_phosphotase          0.161                0.214
de_ritis                      0.084               -0.110
gender                        0.026               -0.031
direct_ratio                  0.015               -0.017
total_proteins                0.012     

In [10]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

# ==================================================
# CHECK DATA
# ==================================================

print("X shape:", X.shape)
print("y shape:", y.shape)

# If y is pandas Series convert to numpy
if hasattr(y, "values"):
    y = y.values

# ==================================================
# NESTED CROSS VALIDATION
# ==================================================

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

inner_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        max_iter=5000,
        early_stopping=True,
        random_state=42
    ))
])

param_grid = {

    "mlp__hidden_layer_sizes": [
        (32,),
        (64,),
        (32, 16)
    ],

    "mlp__activation": [
        "relu",
        "tanh"
    ],

    "mlp__alpha": [
        0.0001,
        0.001,
        0.01
    ],

    "mlp__learning_rate_init": [
        0.001,
        0.01
    ]
}

aucs = []
accs = []
bal_accs = []
mccs = []

sens_list = []
spec_list = []

fold = 1

for train_idx, test_idx in outer_cv.split(X, y):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=inner_cv,
        n_jobs=-1,
        error_score="raise"
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    y_prob = best_model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    auc = roc_auc_score(y_test, y_prob)
    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()

    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    aucs.append(auc)
    accs.append(acc)
    bal_accs.append(bal_acc)
    mccs.append(mcc)

    sens_list.append(sensitivity)
    spec_list.append(specificity)

    print(
        f"Fold {fold}"
        f" | AUC={auc:.3f}"
        f" | ACC={acc:.3f}"
        f" | Best={grid.best_params_}"
    )

    fold += 1

# ==================================================
# FINAL RESULTS
# ==================================================

print("\n" + "=" * 70)
print("NESTED CROSS VALIDATION RESULTS")
print("=" * 70)

print(
    f"AUC           : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}"
)

print(
    f"Accuracy      : {np.mean(accs):.3f} ± {np.std(accs):.3f}"
)

print(
    f"Balanced Acc  : {np.mean(bal_accs):.3f} ± {np.std(bal_accs):.3f}"
)

print(
    f"MCC           : {np.mean(mccs):.3f} ± {np.std(mccs):.3f}"
)

print(
    f"Sensitivity   : {np.mean(sens_list):.3f}"
)

print(
    f"Specificity   : {np.mean(spec_list):.3f}"
)

print("=" * 70)


X shape: (567, 13)
y shape: (567,)
Fold 1 | AUC=0.677 | ACC=0.675 | Best={'mlp__activation': 'tanh', 'mlp__alpha': 0.0001, 'mlp__hidden_layer_sizes': (32, 16), 'mlp__learning_rate_init': 0.01}
Fold 2 | AUC=0.681 | ACC=0.675 | Best={'mlp__activation': 'tanh', 'mlp__alpha': 0.0001, 'mlp__hidden_layer_sizes': (64,), 'mlp__learning_rate_init': 0.01}
Fold 3 | AUC=0.735 | ACC=0.699 | Best={'mlp__activation': 'tanh', 'mlp__alpha': 0.001, 'mlp__hidden_layer_sizes': (32, 16), 'mlp__learning_rate_init': 0.01}
Fold 4 | AUC=0.761 | ACC=0.699 | Best={'mlp__activation': 'tanh', 'mlp__alpha': 0.01, 'mlp__hidden_layer_sizes': (32, 16), 'mlp__learning_rate_init': 0.01}
Fold 5 | AUC=0.733 | ACC=0.735 | Best={'mlp__activation': 'tanh', 'mlp__alpha': 0.01, 'mlp__hidden_layer_sizes': (32,), 'mlp__learning_rate_init': 0.01}

NESTED CROSS VALIDATION RESULTS
AUC           : 0.717 ± 0.033
Accuracy      : 0.697 ± 0.022
Balanced Acc  : 0.541 ± 0.050
MCC           : 0.088 ± 0.129
Sensitivity   : 0.911
Specificity

In [11]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

# ==================================================
# Prepare y
# ==================================================

if hasattr(y, "values"):
    y = y.values

# ==================================================
# Nested CV
# ==================================================

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

inner_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        max_iter=5000,
        random_state=42
    ))
])

param_grid = {
    "lr__C": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ],

    "lr__penalty": [
        "l1",
        "l2"
    ],

    "lr__solver": [
        "liblinear"
    ],

    "lr__class_weight": [
        None,
        "balanced"
    ]
}

aucs = []
accs = []
bal_accs = []
mccs = []

sens_list = []
spec_list = []

fold = 1

for train_idx, test_idx in outer_cv.split(X, y):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=inner_cv,
        n_jobs=-1,
        error_score="raise"
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    y_prob = best_model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    auc = roc_auc_score(y_test, y_prob)
    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()

    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    aucs.append(auc)
    accs.append(acc)
    bal_accs.append(bal_acc)
    mccs.append(mcc)

    sens_list.append(sensitivity)
    spec_list.append(specificity)

    print(
        f"Fold {fold}"
        f" | AUC={auc:.3f}"
        f" | ACC={acc:.3f}"
        f" | Best={grid.best_params_}"
    )

    fold += 1

print("\n" + "="*70)
print("NESTED CV LOGISTIC REGRESSION")
print("="*70)

print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")
print(f"Bal Acc     : {np.mean(bal_accs):.3f} ± {np.std(bal_accs):.3f}")
print(f"MCC         : {np.mean(mccs):.3f} ± {np.std(mccs):.3f}")
print(f"Sensitivity : {np.mean(sens_list):.3f}")
print(f"Specificity : {np.mean(spec_list):.3f}")
print("="*70)

Fold 1 | AUC=0.772 | ACC=0.719 | Best={'lr__C': 1, 'lr__class_weight': None, 'lr__penalty': 'l2', 'lr__solver': 'liblinear'}
Fold 2 | AUC=0.738 | ACC=0.623 | Best={'lr__C': 1, 'lr__class_weight': 'balanced', 'lr__penalty': 'l2', 'lr__solver': 'liblinear'}
Fold 3 | AUC=0.711 | ACC=0.690 | Best={'lr__C': 10, 'lr__class_weight': None, 'lr__penalty': 'l2', 'lr__solver': 'liblinear'}
Fold 4 | AUC=0.756 | ACC=0.699 | Best={'lr__C': 1, 'lr__class_weight': None, 'lr__penalty': 'l2', 'lr__solver': 'liblinear'}
Fold 5 | AUC=0.776 | ACC=0.646 | Best={'lr__C': 10, 'lr__class_weight': 'balanced', 'lr__penalty': 'l2', 'lr__solver': 'liblinear'}

NESTED CV LOGISTIC REGRESSION
AUC         : 0.751 ± 0.024
Accuracy    : 0.676 ± 0.036
Bal Acc     : 0.601 ± 0.065
MCC         : 0.213 ± 0.101
Sensitivity : 0.777
Specificity : 0.426


In [12]:
import os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# ---------- data ----------
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
df = pd.read_csv(os.path.join(OUT_DIR, 'ILPD_cleaned.csv'))
X = df.drop(columns='target')
y = df['target'].values
print(X.shape, 'majority baseline =', round(max(y.mean(), 1 - y.mean()), 3))

SKEW = ['total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase',
        'alt_sgpt', 'ast_sgot', 'de_ritis']
NUM = ['age', 'total_proteins', 'albumin', 'ag_ratio', 'globulin', 'direct_ratio']

def prep():
    return ColumnTransformer([
        ('log', Pipeline([('imp', SimpleImputer(strategy='median')),
                          ('l', FunctionTransformer(np.log1p))]), SKEW),
        ('num', SimpleImputer(strategy='median'), NUM),
        ('bin', 'passthrough', ['gender']),
    ])

def rf():
    return RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=42)

def run(est, X_, y_):
    r = cross_validate(est, X_, y_, cv=cv, scoring=['roc_auc', 'accuracy'])
    return r['test_roc_auc'], r['test_accuracy']

# ---------- A) Paper jaisa: oversample PEHLE, phir CV (leakage ka risk) ----------
Xb, yb = RandomOverSampler(random_state=42).fit_resample(X, y)
Xb = pd.DataFrame(Xb, columns=X.columns)
A = run(Pipeline([('prep', prep()), ('sc', StandardScaler()), ('m', rf())]), Xb, yb)

# ---------- B) Sahi: oversample sirf train fold ke andar ----------
B = run(ImbPipeline([('prep', prep()), ('sc', StandardScaler()),
                     ('over', RandomOverSampler(random_state=42)), ('m', rf())]), X, y)

# ---------- C) Paper ka PCA+FA+LDA integration, sab train fold ke andar ----------
integ = FeatureUnion([('pca', PCA(n_components=0.95)),
                      ('fa', FactorAnalysis(n_components=4, random_state=42)),  # scree se chuno
                      ('lda', LDA())])
C = run(ImbPipeline([('prep', prep()), ('sc', StandardScaler()),
                     ('over', RandomOverSampler(random_state=42)),
                     ('integ', integ), ('m', rf())]), X, y)

for name, (auc, acc) in [('A  leaky (oversample first)', A),
                         ('B  fair (oversample in fold)', B),
                         ('C  fair + PCA/FA/LDA', C)]:
    print(f'{name:30s} AUC = {auc.mean():.3f} ± {auc.std():.3f} | acc = {acc.mean():.3f} ± {acc.std():.3f}')

(567, 13) majority baseline = 0.711
A  leaky (oversample first)    AUC = 0.958 ± 0.017 | acc = 0.855 ± 0.034
B  fair (oversample in fold)   AUC = 0.732 ± 0.066 | acc = 0.695 ± 0.060
C  fair + PCA/FA/LDA           AUC = 0.734 ± 0.061 | acc = 0.695 ± 0.058


In [13]:
"""
LEAKAGE EXPERIMENT (ILPD): oversampling split se PEHLE vs split ke ANDAR
=========================================================================
Sawal: kya published ILPD papers ki high accuracy (jaise Amin et al. 2023: RF ~88%)
sirf is wajah se ho sakti hai ki random oversampling train/test split se pehle lagayi gayi?

Protocols (same data, same 5 models, same preprocessing):
  A1  "Leaky 10-fold"   : poore data ko random-oversample karo -> phir 10-fold CV   (Amin-style)
  A2  "Leaky 75/25"     : poore data ko random-oversample karo -> phir 75/25 split  (Amin-style)
  B   "Correct"         : 10-fold CV, oversampling SIRF train fold ke andar, test fold original
  C   "No resampling"   : 10-fold CV, koi oversampling nahi (class_weight='balanced' jahan available)

Extra diagnostic: dup_overlap = test rows ka kitna hissa train mein EXACT copy ke roop mein maujood hai.
(Leakage ka seedha sabut: A mein bada, B/C mein ~0.)

Padhne ka tareeka:
  - Balanced accuracy aur AUC compare karo (accuracy ka baseline protocols mein alag hai:
    A ka test set balanced hai (baseline 50%), B/C ka original imbalanced (baseline ~71%)).
  - Result jo bhi aaye, honestly report karo. Hypothesis hai, guarantee nahi.

Run: Kaggle pe seedha. Time: ~5-15 min (N_REPEATS=3).
"""
import os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
N_REPEATS = int(os.environ.get('N_REPEATS', 3))
DATA_PATH = os.environ.get(
    'ILPD_PATH',
    '/kaggle/input/datasets/sham1026/indian-liver-patient-dataset/Indian Liver Patient Dataset (ILPD).csv')
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

# ------------------------------------------------------------
# 1. LOAD + CLEAN (v3 jaisa, original 10 features)
# ------------------------------------------------------------
df = pd.read_csv(DATA_PATH)
df.columns = ['age', 'gender', 'total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase',
              'alt_sgpt', 'ast_sgot', 'total_proteins', 'albumin', 'ag_ratio', 'target']
df = df.drop_duplicates().reset_index(drop=True)
df = df[~(df['direct_bilirubin'] > df['total_bilirubin'])].reset_index(drop=True)
df['gender'] = (df['gender'] == 'Male').astype(int)
df['target'] = (df['target'] == 1).astype(int)
X = df.drop(columns='target')
y = df['target'].values
print(f'Rows after cleaning: {len(df)} | patients={y.sum()} healthy={(1 - y).sum()} '
      f'| majority baseline acc={max(y.mean(), 1 - y.mean()):.3f}')

SKEW = ['total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase', 'alt_sgpt', 'ast_sgot']
NUM = ['age', 'total_proteins', 'albumin', 'ag_ratio']

def make_pipeline(model):
    prep = ColumnTransformer([
        ('log', Pipeline([('imp', SimpleImputer(strategy='median')),
                          ('l', FunctionTransformer(np.log1p))]), SKEW),
        ('num', SimpleImputer(strategy='median'), NUM),
        ('bin', 'passthrough', ['gender']),
    ])
    return Pipeline([('prep', prep), ('sc', StandardScaler()), ('m', model)])

# Amin et al. jaise models (koi heavy tuning nahi, taaki sirf protocol ka asar dikhe)
def models(balanced):
    cw = 'balanced' if balanced else None
    return {
        'LR': LogisticRegression(max_iter=3000, class_weight=cw, random_state=SEED),
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'RF': RandomForestClassifier(n_estimators=300, class_weight=cw, random_state=SEED, n_jobs=-1),
        'SVM': SVC(kernel='rbf', class_weight=cw, random_state=SEED),
        'MLP': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1500, random_state=SEED),
    }

# ------------------------------------------------------------
# 2. HELPERS
# ------------------------------------------------------------
def random_oversample(X_, y_, rng):
    """Minority rows ki random copies (with replacement) jab tak classes barabar na ho jaayein."""
    X_ = X_.reset_index(drop=True)
    y_ = np.asarray(y_)
    classes, counts = np.unique(y_, return_counts=True)
    n_max = counts.max()
    idx = []
    for c in classes:
        ci = np.where(y_ == c)[0]
        extra = rng.choice(ci, n_max - len(ci), replace=True) if len(ci) < n_max else np.array([], dtype=int)
        idx.append(np.concatenate([ci, extra]))
    idx = np.concatenate(idx)
    return X_.iloc[idx].reset_index(drop=True), y_[idx]

def dup_overlap(Xtr, Xte):
    """Test rows ka wo hissa jiski exact copy train mein hai."""
    htr = set(pd.util.hash_pandas_object(Xtr, index=False))
    hte = pd.util.hash_pandas_object(Xte, index=False)
    return float(np.mean([h in htr for h in hte]))

def scores(model, Xte):
    return model.predict_proba(Xte)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(Xte)

def evaluate(pipe, Xtr, ytr, Xte, yte):
    m = clone(pipe).fit(Xtr, ytr)
    s = scores(m, Xte)
    pred = m.predict(Xte)
    tn, fp, fn, tp = confusion_matrix(yte, pred, labels=[0, 1]).ravel()
    return {'acc': accuracy_score(yte, pred),
            'bal_acc': balanced_accuracy_score(yte, pred),
            'auc': roc_auc_score(yte, s),
            'sens': tp / (tp + fn) if (tp + fn) else np.nan,
            'spec': tn / (tn + fp) if (tn + fp) else np.nan,
            'dup_overlap': dup_overlap(Xtr, Xte)}

# ------------------------------------------------------------
# 3. RUN ALL PROTOCOLS
# ------------------------------------------------------------
rows = []
for rep in range(N_REPEATS):
    rng = np.random.default_rng(SEED + rep)
    skf = StratifiedKFold(10, shuffle=True, random_state=SEED + rep)

    # --- A: poore data ko pehle oversample (LEAKY) ---
    X_os, y_os = random_oversample(X, y, rng)
    if rep == 0:
        n_dups = int(X_os.duplicated().sum())
        print(f'Oversampled-before-split data: {len(X_os)} rows ({n_dups} exact duplicate rows)')
    mA = models(balanced=False)
    for fold, (tr, te) in enumerate(skf.split(X_os, y_os)):
        for name, mdl in mA.items():
            r = evaluate(make_pipeline(mdl), X_os.iloc[tr], y_os[tr], X_os.iloc[te], y_os[te])
            rows.append({'protocol': 'A1 Leaky 10-fold', 'model': name, 'rep': rep, 'fold': fold, **r})

    sss = StratifiedShuffleSplit(n_splits=10, test_size=0.25, random_state=SEED + rep)
    for fold, (tr, te) in enumerate(sss.split(X_os, y_os)):
        for name, mdl in mA.items():
            r = evaluate(make_pipeline(mdl), X_os.iloc[tr], y_os[tr], X_os.iloc[te], y_os[te])
            rows.append({'protocol': 'A2 Leaky 75/25', 'model': name, 'rep': rep, 'fold': fold, **r})

    # --- B: oversampling sirf train fold ke andar (CORRECT) ---
    # --- C: koi oversampling nahi ---
    mC = models(balanced=True)
    for fold, (tr, te) in enumerate(skf.split(X, y)):
        Xtr, ytr, Xte, yte = X.iloc[tr], y[tr], X.iloc[te], y[te]
        Xtr_os, ytr_os = random_oversample(Xtr, ytr, rng)
        for name in mA:
            r = evaluate(make_pipeline(mA[name]), Xtr_os, ytr_os, Xte, yte)
            rows.append({'protocol': 'B Correct (oversample in-fold)', 'model': name, 'rep': rep, 'fold': fold, **r})
            r = evaluate(make_pipeline(mC[name]), Xtr, ytr, Xte, yte)
            rows.append({'protocol': 'C No resampling', 'model': name, 'rep': rep, 'fold': fold, **r})
    print(f'repeat {rep + 1}/{N_REPEATS} done', flush=True)

res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUT_DIR, 'leakage_experiment_folds.csv'), index=False)

# ------------------------------------------------------------
# 4. SUMMARY
# ------------------------------------------------------------
order = ['A1 Leaky 10-fold', 'A2 Leaky 75/25', 'B Correct (oversample in-fold)', 'C No resampling']
mnames = ['LR', 'KNN', 'RF', 'SVM', 'MLP']

def pivot(metric):
    p = res.groupby(['model', 'protocol'])[metric].mean().unstack('protocol')[order].loc[mnames]
    return p

print('\n' + '=' * 100)
for metric, title in (('bal_acc', 'BALANCED ACCURACY'), ('auc', 'AUC'), ('acc', 'ACCURACY (baseline A: 0.50, B/C: ~0.71)'),
                      ('dup_overlap', 'DUP_OVERLAP (test rows jinki exact copy train mein hai)')):
    print(f'\n{title}')
    print(pivot(metric).round(3).to_string())

gap = (pivot('bal_acc')['A1 Leaky 10-fold'] - pivot('bal_acc')['B Correct (oversample in-fold)']).round(3)
gap_auc = (pivot('auc')['A1 Leaky 10-fold'] - pivot('auc')['B Correct (oversample in-fold)']).round(3)
print('\nInflation due to leakage (A1 minus B):')
print(pd.DataFrame({'bal_acc_gap': gap, 'auc_gap': gap_auc}).to_string())

summary = res.groupby(['protocol', 'model'])[['acc', 'bal_acc', 'auc', 'sens', 'spec', 'dup_overlap']].agg(['mean', 'std']).round(3)
summary.to_csv(os.path.join(OUT_DIR, 'leakage_experiment_summary.csv'))

# ------------------------------------------------------------
# 5. FIGURE
# ------------------------------------------------------------
pb = pivot('bal_acc')
fig, ax = plt.subplots(figsize=(9, 4.5))
w = 0.2
colors = ['#c0392b', '#e67e22', '#27ae60', '#2471a3']
for i, p in enumerate(order):
    ax.bar(np.arange(len(mnames)) + (i - 1.5) * w, pb[p].values, w, label=p, color=colors[i])
ax.axhline(0.5, ls='--', c='gray', lw=0.8)
ax.set_xticks(np.arange(len(mnames))); ax.set_xticklabels(mnames)
ax.set_ylabel('Balanced accuracy'); ax.set_ylim(0.4, 1.0)
ax.set_title('Effect of oversampling BEFORE vs INSIDE cross-validation (ILPD)')
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'leakage_comparison.png'), dpi=200); plt.close()
print('\nSaved: leakage_experiment_folds.csv, leakage_experiment_summary.csv, leakage_comparison.png')

Rows after cleaning: 567 | patients=403 healthy=164 | majority baseline acc=0.711
Oversampled-before-split data: 806 rows (239 exact duplicate rows)
repeat 1/3 done
repeat 2/3 done
repeat 3/3 done


BALANCED ACCURACY
protocol  A1 Leaky 10-fold  A2 Leaky 75/25  B Correct (oversample in-fold)  C No resampling
model                                                                                      
LR                   0.718           0.717                           0.694            0.697
KNN                  0.747           0.725                           0.662            0.605
RF                   0.840           0.821                           0.612            0.569
SVM                  0.729           0.730                           0.693            0.697
MLP                  0.848           0.770                           0.603            0.599

AUC
protocol  A1 Leaky 10-fold  A2 Leaky 75/25  B Correct (oversample in-fold)  C No resampling
model                                     

In [14]:
"""
Final Logistic Regression pipeline rebuild (SHAP script ke liye)
================================================================
v3 ka best model LR tha (C=0.5, balanced). LR deterministic hai, isliye poore data pe wahi model
dobara fit karna v3 ke 10-model nested CV ko dobara chalane jaisa hi nateeja deta hai (aur bahut tez hai).

Zaroorat: /kaggle/working/ILPD_cleaned.csv jisme engineered columns hon
          (de_ritis, direct_ratio, globulin) -- yeh v3 script banati hai.
Output  : /kaggle/working/unimodal_best_model.joblib  (sirf pipeline, SHAP script isi ko padhti hai)
"""
import os, sys
import numpy as np, pandas as pd, joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression

OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
CSV = os.environ.get('ILPD_CLEAN_PATH', os.path.join(OUT_DIR, 'ILPD_cleaned.csv'))

df = pd.read_csv(CSV)
SKEW = ['total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase', 'alt_sgpt', 'ast_sgot', 'de_ritis']
NUM = ['age', 'total_proteins', 'albumin', 'ag_ratio', 'globulin', 'direct_ratio']
missing = [c for c in SKEW + NUM + ['gender', 'target'] if c not in df.columns]
if missing:
    sys.exit(f'ILPD_cleaned.csv mein ye columns nahi hain: {missing}\n'
             'Yeh purani (v1) file hai. Pehle v3 script run karo, woh engineered columns ke saath file dobara banati hai.')

X, y = df.drop(columns='target'), df['target'].values
prep = ColumnTransformer([
    ('log', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('l', FunctionTransformer(np.log1p))]), SKEW),
    ('num', SimpleImputer(strategy='median'), NUM),
    ('bin', 'passthrough', ['gender']),
])
model = Pipeline([('prep', prep), ('sc', StandardScaler()),
                  ('m', LogisticRegression(C=0.5, max_iter=3000, class_weight='balanced', random_state=42))])
model.fit(X, y)

out = os.path.join(OUT_DIR, 'unimodal_best_model.joblib')
joblib.dump(model, out)
print(f'Saved {out} | rows={len(X)}, features={X.shape[1]}')
print('Training AUC (sirf sanity check, performance claim nahi):',
      round(__import__("sklearn.metrics", fromlist=["roc_auc_score"]).roc_auc_score(y, model.predict_proba(X)[:, 1]), 3))

Saved /kaggle/working/unimodal_best_model.joblib | rows=567, features=13
Training AUC (sirf sanity check, performance claim nahi): 0.773


In [15]:
"""
Final Logistic Regression pipeline rebuild (SHAP script ke liye)
================================================================
Ye script khud sab sambhalti hai:
  1) Agar ILPD_cleaned.csv purani (v1) hai, to raw ILPD se v3 jaisi cleaning + engineered features
     dobara banakar ILPD_cleaned.csv (567 rows, 14 columns) save karti hai (v3 ka 10-model CV chalane ki zarurat nahi).
  2) Final Logistic Regression (C=0.5, balanced) poore data pe fit karke unimodal_best_model.joblib save karti hai.
LR deterministic hai, isliye yeh v3 ke final LR model jaisa hi hai.

Uske baad SHAP (LinearExplainer) script run karo.
"""
import os, sys
import numpy as np, pandas as pd, joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
CSV = os.environ.get('ILPD_CLEAN_PATH', os.path.join(OUT_DIR, 'ILPD_cleaned.csv'))
RAW = os.environ.get(
    'ILPD_PATH',
    '/kaggle/input/datasets/sham1026/indian-liver-patient-dataset/Indian Liver Patient Dataset (ILPD).csv')

SKEW = ['total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase', 'alt_sgpt', 'ast_sgot', 'de_ritis']
NUM = ['age', 'total_proteins', 'albumin', 'ag_ratio', 'globulin', 'direct_ratio']
NEEDED = SKEW + NUM + ['gender', 'target']

def rebuild_cleaned():
    d = pd.read_csv(RAW)
    d.columns = ['age', 'gender', 'total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase',
                 'alt_sgpt', 'ast_sgot', 'total_proteins', 'albumin', 'ag_ratio', 'target']
    d = d.drop_duplicates().reset_index(drop=True)                        # 583 -> 570
    d = d[~(d['direct_bilirubin'] > d['total_bilirubin'])].reset_index(drop=True)   # 570 -> 567
    d['gender'] = (d['gender'] == 'Male').astype(int)
    d['target'] = (d['target'] == 1).astype(int)
    d['de_ritis'] = d['ast_sgot'] / (d['alt_sgpt'] + 1)
    d['direct_ratio'] = d['direct_bilirubin'] / (d['total_bilirubin'] + 0.1)
    d['globulin'] = d['total_proteins'] - d['albumin']
    return d

df = pd.read_csv(CSV) if os.path.exists(CSV) else None
if df is None or any(c not in df.columns for c in NEEDED):
    if not os.path.exists(RAW):
        sys.exit(f'Raw ILPD file nahi mili: {RAW}\nILPD_PATH set karo ya Kaggle input ka path check karo.')
    df = rebuild_cleaned()
    df.to_csv(CSV, index=False)
    print(f'ILPD_cleaned.csv dobara banayi (v3 jaisi cleaning): {len(df)} rows, {df.shape[1]} columns -> {CSV}')
    if len(df) != 567:
        print(f'WARNING: 567 rows expected, {len(df)} mile. Raw file/cleaning check karo.')
else:
    print(f'Existing ILPD_cleaned.csv sahi hai: {len(df)} rows, {df.shape[1]} columns')

X, y = df.drop(columns='target'), df['target'].values
prep = ColumnTransformer([
    ('log', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('l', FunctionTransformer(np.log1p))]), SKEW),
    ('num', SimpleImputer(strategy='median'), NUM),
    ('bin', 'passthrough', ['gender']),
])
model = Pipeline([('prep', prep), ('sc', StandardScaler()),
                  ('m', LogisticRegression(C=0.5, max_iter=3000, class_weight='balanced', random_state=42))])
model.fit(X, y)

out = os.path.join(OUT_DIR, 'unimodal_best_model.joblib')
joblib.dump(model, out)
print(f'Saved {out} | rows={len(X)}, features={X.shape[1]}')
print('Training AUC (sirf sanity check, performance claim nahi):',
      round(roc_auc_score(y, model.predict_proba(X)[:, 1]), 3))

Existing ILPD_cleaned.csv sahi hai: 567 rows, 14 columns
Saved /kaggle/working/unimodal_best_model.joblib | rows=567, features=13
Training AUC (sirf sanity check, performance claim nahi): 0.773


In [16]:
"""
ILPD EXTRAS - paper ko mazboot banane wale 5 modules (ek hi script)
====================================================================
A  Sensitivity: 3 'impossible' records (direct bilirubin > total bilirubin) rakhne / cap karne / hatane ka asar
B  Leakage ke prakar (paired, same folds): duplicates kept, scaling-on-all-data, feature-selection-on-all-data
   (+ control: feature selection fold ke ANDAR, taaki features ghatane ka asar leakage se alag dikhe)
C  Learning curve (training size badhane par AUC)
D  Calibration (Brier, ECE, slope/intercept, reliability plot) + sex-wise performance (AUC, FNR, FPR, bootstrap CI)
E  Explainability: permutation importance (fold ke bahar wale data pe) + TreeSHAP (agar shap installed ho)

Chalane ka tareeka (Kaggle):
  - Poora script paste karke run karo. Sirf kuch modules chahiye to cell ke top par:
        import os; os.environ['MODULES'] = 'A,B'
  - Jaldi test ke liye: os.environ['N_REPEATS'] = '1'
  - Dataset auto-detect hota hai (/kaggle/input ke neeche). Na mile to os.environ['ILPD_PATH'] set karo.

Sab outputs /kaggle/working mein save hote hain (CSV + PNG). Result jo bhi aaye, honestly report karo.
"""
import os, glob, warnings, traceback
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

from scipy import stats
from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score, brier_score_loss, balanced_accuracy_score
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
N_REPEATS = int(os.environ.get('N_REPEATS', 3))
MODULES = set(os.environ.get('MODULES', 'A,B,C,D,E').replace(' ', '').upper().split(','))
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

# ------------------------------------------------------------
# DATA
# ------------------------------------------------------------
COLS = ['age', 'gender', 'total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase',
        'alt_sgpt', 'ast_sgot', 'total_proteins', 'albumin', 'ag_ratio', 'target']

def find_ilpd():
    p = os.environ.get('ILPD_PATH')
    if p and os.path.exists(p):
        return p
    cands = glob.glob('/kaggle/input/**/*.csv', recursive=True)
    for c in cands:
        if 'ilpd' in c.lower() or 'liver' in c.lower():
            return c
    raise SystemExit('ILPD CSV nahi mili. Notebook mein dataset Input ke roop mein jodo ya os.environ["ILPD_PATH"] set karo.\n'
                     f'/kaggle/input mein mili CSVs: {cands}')

path = find_ilpd()
print('Using data file:', path)
raw = pd.read_csv(path)
raw.columns = COLS
raw['gender'] = (raw['gender'] == 'Male').astype(int)
raw['target'] = (raw['target'] == 1).astype(int)

def add_features(d):
    d = d.copy()
    d['de_ritis'] = d['ast_sgot'] / (d['alt_sgpt'] + 1)
    d['direct_ratio'] = d['direct_bilirubin'] / (d['total_bilirubin'] + 0.1)
    d['globulin'] = d['total_proteins'] - d['albumin']
    return d

dup_mask = raw.duplicated(keep='first')
uniq = raw[~dup_mask].reset_index(drop=True)          # 570 unique rows
extras = raw[dup_mask].copy()                         # 13 extra copies
bad_u = uniq['direct_bilirubin'] > uniq['total_bilirubin']
base = add_features(uniq[~bad_u].reset_index(drop=True))
extras_ok = add_features(extras[~(extras['direct_bilirubin'] > extras['total_bilirubin'])])
X = base.drop(columns='target')
y = base['target'].values
print(f'raw={len(raw)} | unique={len(uniq)} | impossible={int(bad_u.sum())} | analysed={len(base)} '
      f'| patients={y.sum()} healthy={(1 - y).sum()} | majority baseline={max(y.mean(), 1 - y.mean()):.3f}')

SKEW = ['total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase', 'alt_sgpt', 'ast_sgot', 'de_ritis']
NUM = ['age', 'total_proteins', 'albumin', 'ag_ratio', 'globulin', 'direct_ratio']
BIN = ['gender']
FEATS = SKEW + NUM + BIN                                # model ke andar ka feature order

def make_prep():
    return ColumnTransformer([
        ('log', Pipeline([('imp', SimpleImputer(strategy='median')),
                          ('l', FunctionTransformer(np.log1p))]), SKEW),
        ('num', SimpleImputer(strategy='median'), NUM),
        ('bin', 'passthrough', BIN),
    ])

def make_pipe(model):
    return Pipeline([('prep', make_prep()), ('sc', StandardScaler()), ('m', model)])

def get_models(n_trees=200):
    return {
        'LR': LogisticRegression(C=0.5, max_iter=3000, class_weight='balanced', random_state=SEED),
        'SVM': SVC(C=1, gamma='scale', class_weight='balanced', random_state=SEED),
        'KNN': KNeighborsClassifier(n_neighbors=15),
        'RF': RandomForestClassifier(n_estimators=n_trees, max_depth=6, min_samples_leaf=5,
                                     class_weight='balanced_subsample', random_state=SEED, n_jobs=-1),
        'ExtraTrees': ExtraTreesClassifier(n_estimators=n_trees, min_samples_leaf=4, max_features='sqrt',
                                           class_weight='balanced_subsample', random_state=SEED, n_jobs=-1),
    }

def score(m, Xe):
    return m.predict_proba(Xe)[:, 1] if hasattr(m, 'predict_proba') else m.decision_function(Xe)

def take(A, idx):
    return A.iloc[idx] if hasattr(A, 'iloc') else A[idx]

def cv_auc(est, A, yy, splits, extra=None):
    out = []
    for tr, te in splits:
        Xtr, ytr = take(A, tr), yy[tr]
        if extra is not None:                         # extra = (DataFrame, labels) sirf TRAIN mein jodta hai
            Xtr = pd.concat([Xtr, extra[0]], ignore_index=True)
            ytr = np.concatenate([ytr, extra[1]])
        m = clone(est).fit(Xtr, ytr)
        out.append(roc_auc_score(yy[te], score(m, take(A, te))))
    return np.array(out)

def corrected_ttest(a, b, n_train, n_test):
    d = np.asarray(a) - np.asarray(b)
    k = len(d)
    var = np.var(d, ddof=1) * (1 / k + n_test / n_train)
    if var <= 0:
        return d.mean(), 1.0
    t = d.mean() / np.sqrt(var)
    return d.mean(), 2 * stats.t.sf(abs(t), df=k - 1)

def run(tag, fn):
    if tag not in MODULES:
        return
    print('\n' + '=' * 100 + f'\nMODULE {tag}\n' + '=' * 100)
    try:
        fn()
    except Exception:
        print(f'MODULE {tag} fail hua (baaki modules chalte rahenge):')
        traceback.print_exc()

# ------------------------------------------------------------
# A. Sensitivity: 3 'impossible' records
# ------------------------------------------------------------
def module_A():
    bad_rows = uniq[bad_u][['total_bilirubin', 'direct_bilirubin', 'target']]
    print('Impossible rows (direct > total bilirubin):')
    print(bad_rows.to_string())
    print('(Agar difference bahut chhota hai, to rounding/entry error ho sakta hai.)\n')

    keep = add_features(uniq.copy())
    cap = uniq.copy()
    cap.loc[bad_u, 'direct_bilirubin'] = cap.loc[bad_u, 'total_bilirubin']
    cap = add_features(cap)
    variants = {'remove (analysed set)': base, 'keep as is': keep, 'cap DB at TB': cap}
    models = {k: v for k, v in get_models().items() if k in ('LR', 'SVM', 'RF', 'ExtraTrees')}
    rows = []
    for vname, d in variants.items():
        Xv, yv = d.drop(columns='target'), d['target'].values
        spl = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REPEATS, random_state=SEED).split(Xv, yv))
        for mname, mdl in models.items():
            a = cv_auc(make_pipe(mdl), Xv, yv, spl)
            rows.append({'variant': vname, 'model': mname, 'n': len(d), 'AUC': a.mean(), 'SD': a.std()})
    res = pd.DataFrame(rows)
    piv = res.pivot(index='model', columns='variant', values='AUC').round(3)
    print('Mean AUC (repeated 5-fold):')
    print(piv.to_string())
    print(f"\nFold SD ~ {res['SD'].mean():.3f}. Row sets alag hain (3 rows), isliye yeh descriptive tulna hai, paired test nahi.")
    res.to_csv(os.path.join(OUT_DIR, 'extras_A_impossible_records.csv'), index=False)

# ------------------------------------------------------------
# B. Leakage ke prakar (same folds, paired)
# ------------------------------------------------------------
def module_B():
    splits = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REPEATS, random_state=SEED).split(X, y))
    n_test = len(y) // 5
    n_train = len(y) - n_test
    prep_all = Pipeline([('prep', make_prep()), ('sc', StandardScaler())]).fit(X)       # poore data pe (unsupervised)
    Xa = prep_all.transform(X)
    sel_all = SelectKBest(f_classif, k=5).fit(Xa, y)                                    # poore data + labels pe
    Xs = sel_all.transform(Xa)
    print('Feature selection (k=5, poore data pe) ne chuna:', [FEATS[i] for i in sel_all.get_support(indices=True)])
    y_extra = extras_ok['target'].values
    X_extra = extras_ok.drop(columns='target')
    print(f'Duplicates variant: train mein {len(extras_ok)} extra copies jodi gayi (test mein sirf unique rows).\n')

    models = {k: v for k, v in get_models(n_trees=150).items() if k in ('LR', 'KNN', 'SVM', 'RF')}
    rows = []
    for name, mdl in models.items():
        r0 = cv_auc(make_pipe(mdl), X, y, splits)                                             # R0 sahi tareeka
        r1 = cv_auc(make_pipe(mdl), X, y, splits, extra=(X_extra, y_extra))                   # R1 duplicates train mein
        r2 = cv_auc(clone(mdl), Xa, y, splits)                                                # R2 scaling/imputation poore data pe
        r3 = cv_auc(clone(mdl), Xs, y, splits)                                                # R3 feature selection poore data pe
        r3b = cv_auc(Pipeline([('sel', SelectKBest(f_classif, k=5)), ('m', clone(mdl))]), Xa, y, splits)  # R3b selection fold ke andar
        for tag, arr, ref, ref_tag in [('R1 duplicates kept in train', r1, r0, 'R0'),
                                       ('R2 scale/impute on all data', r2, r0, 'R0'),
                                       ('R3 select-5 on all data', r3, r3b, 'R3b'),
                                       ('R3b select-5 inside folds (control)', r3b, r0, 'R0')]:
            d, p = corrected_ttest(arr, ref, n_train, n_test)
            rows.append({'model': name, 'variant': tag, 'AUC': arr.mean(), 'ref': ref_tag,
                         'dAUC_vs_ref': d, 'p_corrected': p})
        rows.append({'model': name, 'variant': 'R0 correct (reference)', 'AUC': r0.mean(),
                     'ref': '-', 'dAUC_vs_ref': 0.0, 'p_corrected': np.nan})
    res = pd.DataFrame(rows)
    res.to_csv(os.path.join(OUT_DIR, 'extras_B_leakage_types.csv'), index=False)
    show = res.copy()
    show['AUC'] = show['AUC'].round(3)
    show['dAUC_vs_ref'] = show['dAUC_vs_ref'].map(lambda v: f'{v:+.3f}')
    show['p_corrected'] = show['p_corrected'].round(3)
    print(show.to_string(index=False))
    print('\nPadhne ka tareeka:')
    print(' - R3 ka asli leakage-asar = R3 vs R3b (dono mein 5 features; sirf selection ka tarika alag).')
    print(' - R3b vs R0 = sirf features ghatane (13 -> 5) ka asar, leakage nahi.')
    print(' - Fold SD ~0.06 hai; |dAUC| < 0.02 ko asli asar mat maano jab tak p_corrected < 0.05 na ho.')

# ------------------------------------------------------------
# C. Learning curve
# ------------------------------------------------------------
def module_C():
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REPEATS, random_state=SEED)
    sizes = np.array([0.2, 0.35, 0.5, 0.65, 0.8, 1.0])
    models = {k: v for k, v in get_models(n_trees=150).items() if k in ('LR', 'SVM', 'RF')}
    fig, axes = plt.subplots(1, len(models), figsize=(4.2 * len(models), 3.8), sharey=True)
    rows = []
    for ax, (name, mdl) in zip(np.atleast_1d(axes), models.items()):
        ts, tr_s, va_s = learning_curve(make_pipe(mdl), X, y, train_sizes=sizes, cv=cv,
                                        scoring='roc_auc', n_jobs=-1)
        for s, a, b in zip(ts, tr_s, va_s):
            rows.append({'model': name, 'train_size': int(s), 'train_AUC': a.mean(),
                         'val_AUC': b.mean(), 'val_SD': b.std()})
        ax.plot(ts, tr_s.mean(1), 'o-', label='train', color='#c0392b')
        ax.errorbar(ts, va_s.mean(1), yerr=va_s.std(1), fmt='o-', label='validation', color='#2471a3', capsize=3)
        ax.set_title(name); ax.set_xlabel('Training records'); ax.grid(alpha=.3)
    np.atleast_1d(axes)[0].set_ylabel('AUC'); np.atleast_1d(axes)[0].legend()
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'extras_C_learning_curve.png'), dpi=200); plt.close()
    res = pd.DataFrame(rows)
    res.to_csv(os.path.join(OUT_DIR, 'extras_C_learning_curve.csv'), index=False)
    print(res.round(3).to_string(index=False))
    for name in models:
        g = res[res.model == name].reset_index(drop=True)
        print(f"{name}: val AUC {g.val_AUC.iloc[2]:.3f} (n={g.train_size.iloc[2]}) -> {g.val_AUC.iloc[-1]:.3f} (n={g.train_size.iloc[-1]}); "
              f"gain from last 50% data = {g.val_AUC.iloc[-1] - g.val_AUC.iloc[2]:+.3f}")
    print('\nNote: curve flat ho to bhi sirf n<=454 tak ka claim kar sakte ho; usse zyada data ke baare mein extrapolate mat karo.')

# ------------------------------------------------------------
# D. Calibration + sex-wise performance
# ------------------------------------------------------------
def oof_probs(est, n_rep):
    P = np.zeros((n_rep, len(y)))
    for r in range(n_rep):
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED + r).split(X, y):
            m = clone(est).fit(X.iloc[tr], y[tr])
            P[r, te] = m.predict_proba(X.iloc[te])[:, 1]
    return P

def calib_stats(yy, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    q = pd.qcut(p, 10, duplicates='drop')
    g = pd.DataFrame({'y': yy, 'p': p, 'q': q}).groupby('q', observed=True).agg(y=('y', 'mean'), p=('p', 'mean'), n=('y', 'size'))
    ece = float((g['n'] / g['n'].sum() * (g['y'] - g['p']).abs()).sum())
    lg = np.log(p / (1 - p)).reshape(-1, 1)
    lr = LogisticRegression(C=1e6, max_iter=2000).fit(lg, yy)
    return {'AUC': roc_auc_score(yy, p), 'Brier': brier_score_loss(yy, p), 'ECE': ece,
            'slope': float(lr.coef_[0, 0]), 'intercept': float(lr.intercept_[0])}

def module_D():
    ests = {
        'LR (unweighted)': make_pipe(LogisticRegression(C=0.5, max_iter=3000, random_state=SEED)),
        'LR (balanced)': make_pipe(LogisticRegression(C=0.5, max_iter=3000, class_weight='balanced', random_state=SEED)),
        'RF (balanced)': make_pipe(RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=5,
                                                          class_weight='balanced_subsample', random_state=SEED, n_jobs=-1)),
    }
    P = {k: oof_probs(e, N_REPEATS) for k, e in ests.items()}
    rows = []
    for k, arr in P.items():
        st = pd.DataFrame([calib_stats(y, arr[r]) for r in range(arr.shape[0])])
        rows.append({'model': k, **{c: f'{st[c].mean():.3f} +/- {st[c].std(ddof=0):.3f}' for c in st.columns}})
    tab = pd.DataFrame(rows)
    print('Calibration (out-of-fold, repeated 5-fold; ideal: slope=1, intercept=0, ECE/Brier chhota):')
    print(tab.to_string(index=False))
    print(f'Prevalence = {y.mean():.3f}; Brier ka "sabko prevalence bolo" baseline = {y.mean() * (1 - y.mean()):.3f}')
    print('Note: class_weight="balanced" probabilities ko shift karta hai, isliye calibration ke liye unweighted LR dekho.\n')
    tab.to_csv(os.path.join(OUT_DIR, 'extras_D_calibration.csv'), index=False)

    fig, ax = plt.subplots(figsize=(4.6, 4.4))
    ax.plot([0, 1], [0, 1], 'k--', lw=.8, label='perfect')
    for k, arr in P.items():
        fo, mp = calibration_curve(y, arr.mean(0), n_bins=8, strategy='quantile')
        ax.plot(mp, fo, 'o-', label=k)
    ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed fraction of liver patients')
    ax.set_title('Reliability (out-of-fold)'); ax.legend(fontsize=7); ax.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'extras_D_calibration.png'), dpi=200); plt.close()

    # ---- sex-wise (final model jaisa LR balanced; threshold = pooled OOF pe balanced accuracy maximize, descriptive) ----
    pm = P['LR (balanced)'].mean(0)
    grid = np.linspace(0.2, 0.8, 61)
    thr = grid[int(np.argmax([balanced_accuracy_score(y, (pm >= t).astype(int)) for t in grid]))]
    rng = np.random.default_rng(SEED)
    sex = X['gender'].values
    out, boots = [], {}
    for lab, name in ((0, 'Female'), (1, 'Male')):
        idx = np.where(sex == lab)[0]
        yy, pp = y[idx], pm[idx]
        pred = (pp >= thr).astype(int)
        tp = ((pred == 1) & (yy == 1)).sum(); fn = ((pred == 0) & (yy == 1)).sum()
        tn = ((pred == 0) & (yy == 0)).sum(); fp = ((pred == 1) & (yy == 0)).sum()
        bs = []
        for _ in range(1000):
            j = rng.integers(0, len(idx), len(idx))
            if len(np.unique(yy[j])) == 2:
                bs.append(roc_auc_score(yy[j], pp[j]))
        boots[name] = np.array(bs)
        out.append({'sex': name, 'n': len(idx), 'patient_rate': yy.mean(), 'AUC': roc_auc_score(yy, pp),
                    'AUC_95CI': f'{np.percentile(bs, 2.5):.3f}-{np.percentile(bs, 97.5):.3f}',
                    'sensitivity': tp / (tp + fn), 'FNR': fn / (tp + fn), 'specificity': tn / (tn + fp), 'FPR': fp / (tn + fp)})
    res = pd.DataFrame(out)
    print(f'Sex-wise performance (LR balanced, OOF probs averaged over repeats, threshold={thr:.2f}):')
    print(res.round(3).to_string(index=False))
    m = min(len(boots['Male']), len(boots['Female']))
    diff = boots['Male'][:m] - boots['Female'][:m]
    print(f'AUC(Male) - AUC(Female) = {res.AUC[1] - res.AUC[0]:+.3f}, bootstrap 95% CI [{np.percentile(diff, 2.5):+.3f}, {np.percentile(diff, 97.5):+.3f}]')
    print(f'FNR gap (Male - Female) = {res.FNR[1] - res.FNR[0]:+.3f}  (Straw & Wu jaisa disparity check; chhote female subgroup mein CI chaudi hogi)')
    res.to_csv(os.path.join(OUT_DIR, 'extras_D_sexwise.csv'), index=False)

# ------------------------------------------------------------
# E. Explainability: permutation importance + TreeSHAP
# ------------------------------------------------------------
def module_E():
    models = {k: v for k, v in get_models(n_trees=200).items() if k in ('LR', 'SVM', 'RF', 'ExtraTrees')}
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=max(1, N_REPEATS - 1), random_state=SEED)
    acc = {k: [] for k in models}
    for tr, te in cv.split(X, y):
        for name, mdl in models.items():
            est = make_pipe(mdl).fit(X.iloc[tr], y[tr])
            pi = permutation_importance(est, X.iloc[te], y[te], scoring='roc_auc', n_repeats=10,
                                        random_state=SEED, n_jobs=-1)
            acc[name].append(pi.importances_mean)
    imp = pd.DataFrame({k: np.mean(v, axis=0) for k, v in acc.items()}, index=X.columns)
    imp = imp.sort_values('LR', ascending=False)
    print('Permutation importance (drop in AUC jab feature shuffle ho; fold ke BAHAR wale data pe):')
    print(imp.round(4).to_string())
    print('\nRank correlation (Spearman) models ke beech:')
    print(imp.corr(method='spearman').round(2).to_string())
    imp.to_csv(os.path.join(OUT_DIR, 'extras_E_permutation_importance.csv'))
    ax = imp.iloc[::-1].plot.barh(figsize=(7, 5.5), width=.8)
    ax.set_xlabel('Mean AUC drop'); ax.set_title('Permutation importance (out-of-fold)')
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'extras_E_permutation_importance.png'), dpi=200); plt.close()
    print('Note: correlated features (TB-DB, ALT-AST, TP-ALB-globulin) ka importance aapas mein baat jaata hai; groups mein padho.')

    try:
        import shap
    except Exception:
        print('\nshap installed nahi hai, TreeSHAP skip. (Kaggle par aam taur par pre-installed hota hai.)')
        return
    names = SKEW + NUM + BIN
    for name in ('RF', 'ExtraTrees'):
        est = make_pipe(models[name]).fit(X, y)
        Xt = est[:-1].transform(X)
        sv = shap.TreeExplainer(est[-1]).shap_values(Xt)
        sv = np.array(sv[1]) if isinstance(sv, list) else np.array(sv)
        if sv.ndim == 3:
            sv = sv[:, :, 1]
        plt.figure(); shap.summary_plot(sv, pd.DataFrame(Xt, columns=names), show=False)
        plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, f'extras_E_shap_{name}.png'), dpi=200, bbox_inches='tight'); plt.close()
        ms = pd.Series(np.abs(sv).mean(0), index=names).sort_values(ascending=False)
        ms.to_csv(os.path.join(OUT_DIR, f'extras_E_shap_{name}.csv'))
        print(f'\nTreeSHAP mean |SHAP| ({name}, poore data pe fit; explanation ke liye, performance estimate nahi):')
        print(ms.round(3).to_string())

run('A', module_A)
run('B', module_B)
run('C', module_C)
run('D', module_D)
run('E', module_E)
print('\nDone. Files: extras_*.csv / extras_*.png (/kaggle/working).')

Using data file: /kaggle/input/datasets/sham1026/indian-liver-patient-dataset/Indian Liver Patient Dataset (ILPD).csv
raw=583 | unique=570 | impossible=3 | analysed=567 | patients=403 healthy=164 | majority baseline=0.711

MODULE A
Impossible rows (direct > total bilirubin):
     total_bilirubin  direct_bilirubin  target
233              1.8               9.0       1
248              1.5               7.0       1
266              1.0               1.4       1
(Agar difference bahut chhota hai, to rounding/entry error ho sakta hai.)

Mean AUC (repeated 5-fold):
variant     cap DB at TB  keep as is  remove (analysed set)
model                                                      
ExtraTrees         0.750       0.746                  0.744
LR                 0.750       0.750                  0.750
RF                 0.734       0.734                  0.734
SVM                0.747       0.745                  0.742

Fold SD ~ 0.032. Row sets alag hain (3 rows), isliye yeh descriptive tul

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

def load_bupa_data():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/liver-disorders/bupa.data"
    column_names = ['mcv', 'alkphos', 'sgpt', 'sgot', 'gammagt', 'drinks', 'selector']
    df = pd.read_csv(url, header=None, names=column_names)
    
    # Correct target definition using the selector field instead of leaking 'drinks'
    # (Selector 1 typically denotes the disordered group, Selector 2 the control)
    df['target'] = (df['selector'] == 1).astype(int)
    
    # Drop selector and target from features
    X = df.drop(columns=['selector', 'target'])
    y = df['target'].values
    return X, y

def main():
    # 1. Load Data
    X, y = load_bupa_data()
    print(f"Dataset shape: {X.shape[0]} rows, {X.shape[1]} features.")
    print(f"Class distribution: Positive = {sum(y)}, Negative = {len(y) - sum(y)}\n")
    
    # 2. Define Classifiers with Pipeline (incorporating scaling to prevent data leakage)
    models = {
        "Logistic Regression": Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=1000, random_state=42))
        ]),
        "Support Vector Machine": Pipeline([
            ('scaler', StandardScaler()),
            ('clf', SVC(probability=True, random_state=42))
        ]),
        "Random Forest": Pipeline([
            ('scaler', StandardScaler()),
            ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
        ])
    }
    
    # 3. Setup Cross-Validation
    cv = StratifiedKDirect = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) if 'StratifiedKDirect' else StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    scoring_metrics = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'roc_auc': 'roc_auc'
    }
    
    # 4. Evaluate Models
    results = []
    
    for name, pipeline in models.items():
        cv_results = cross_validate(pipeline, X, y, cv=cv, scoring=scoring_metrics)
        
        results.append({
            'Model': name,
            'Accuracy': f"{np.mean(cv_results['test_accuracy']):.3f} ± {np.std(cv_results['test_accuracy']):.3f}",
            'Precision': f"{np.mean(cv_results['test_precision']):.3f} ± {np.std(cv_results['test_precision']):.3f}",
            'Recall (Sensitivity)': f"{np.mean(cv_results['test_recall']):.3f} ± {np.std(cv_results['test_recall']):.3f}",
            'F1-Score': f"{np.mean(cv_results['test_f1']):.3f} ± {np.std(cv_results['test_f1']):.3f}",
            'ROC-AUC': f"{np.mean(cv_results['test_roc_auc']):.3f} ± {np.std(cv_results['test_roc_auc']):.3f}"
        })
        
    # 5. Display Summary Table
    results_df = pd.DataFrame(results)
    print("--- BUPA Dataset Cross-Validation Performance ---")
    print(results_df.to_string(index=False))

if __name__ == "__main__":
    main()

Dataset shape: 345 rows, 6 features.
Class distribution: Positive = 145, Negative = 200

--- BUPA Dataset Cross-Validation Performance ---
                 Model      Accuracy     Precision Recall (Sensitivity)      F1-Score       ROC-AUC
   Logistic Regression 0.675 ± 0.076 0.651 ± 0.111        0.524 ± 0.110 0.574 ± 0.094 0.714 ± 0.092
Support Vector Machine 0.701 ± 0.046 0.713 ± 0.097        0.524 ± 0.122 0.589 ± 0.080 0.737 ± 0.090
         Random Forest 0.722 ± 0.067 0.712 ± 0.096        0.586 ± 0.105 0.637 ± 0.086 0.751 ± 0.079


In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

def load_bupa_data():
    """
    Load the BUPA Liver Disorders dataset cleanly without target leakage.
    """
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/liver-disorders/bupa.data"
    column_names = ['mcv', 'alkphos', 'sgpt', 'sgot', 'gammagt', 'drinks', 'selector']
    print("Fetching BUPA Liver Disorders dataset...")
    df = pd.read_csv(url, header=None, names=column_names)
    
    # Define binary target using the selector field
    df['target'] = (df['selector'] == 1).astype(int)
    
    X = df.drop(columns=['selector', 'target'])
    y = df['target'].values
    return X, y

def run_nested_cv_benchmark():
    X, y = load_bupa_data()
    print(f"Dataset shape: {X.shape[0]} rows, {X.shape[1]} features.")
    print(f"Class distribution: Positive = {sum(y)}, Negative = {len(y) - sum(y)}\n")
    
    # Define base pipelines
    rf_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(random_state=42))
    ])
    
    gb_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(random_state=42))
    ])
    
    # Define hyperparameter grids for tuning (Inner Loop search spaces)
    rf_param_grid = {
        'clf__n_estimators': [50, 100, 200],
        'clf__max_depth': [3, 5, 10, None],
        'clf__min_samples_split': [2, 5, 10]
    }
    
    gb_param_grid = {
        'clf__n_estimators': [50, 100, 200],
        'clf__learning_rate': [0.01, 0.1, 0.2],
        'clf__max_depth': [3, 5, 7]
    }
    
    # Setup Nested Cross-Validation Architecture
    # Inner loop for hyperparameter tuning, Outer loop for unbiased performance estimation
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    models_to_tune = [
        ("Tuned Random Forest", rf_pipeline, rf_param_grid),
        ("Tuned Gradient Boosting", gb_pipeline, gb_param_grid)
    ]
    
    scoring_metrics = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'roc_auc': 'roc_auc'
    }
    
    results = []
    
    print("Running Nested Cross-Validation Grid Search (this may take a few seconds)...")
    for name, pipeline, param_grid in models_to_tune:
        # GridSearchCV automatically handles the inner loop optimization for each outer fold
        nested_grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=inner_cv,
            scoring='roc_auc',
            n_jobs=-1
        )
        
        # Evaluate via outer cross-validation loop
        cv_results = cross_validate(nested_grid_search, X, y, cv=outer_cv, scoring=scoring_metrics)
        
        results.append({
            'Model': name,
            'Accuracy': f"{np.mean(cv_results['test_accuracy']):.3f} ± {np.std(cv_results['test_accuracy']):.3f}",
            'Precision': f"{np.mean(cv_results['test_precision']):.3f} ± {np.std(cv_results['test_precision']):.3f}",
            'Recall (Sensitivity)': f"{np.mean(cv_results['test_recall']):.3f} ± {np.std(cv_results['test_recall']):.3f}",
            'F1-Score': f"{np.mean(cv_results['test_f1']):.3f} ± {np.std(cv_results['test_f1']):.3f}",
            'ROC-AUC': f"{np.mean(cv_results['test_roc_auc']):.3f} ± {np.std(cv_results['test_roc_auc']):.3f}"
        })
        
    results_df = pd.DataFrame(results)
    print("\n--- Nested CV Performance (Fully Tuned Models) ---")
    print(results_df.to_string(index=False))

if __name__ == "__main__":
    run_nested_cv_benchmark()

Fetching BUPA Liver Disorders dataset...
Dataset shape: 345 rows, 6 features.
Class distribution: Positive = 145, Negative = 200

Running Nested Cross-Validation Grid Search (this may take a few seconds)...

--- Nested CV Performance (Fully Tuned Models) ---
                  Model      Accuracy     Precision Recall (Sensitivity)      F1-Score       ROC-AUC
    Tuned Random Forest 0.710 ± 0.077 0.711 ± 0.128        0.545 ± 0.128 0.609 ± 0.106 0.739 ± 0.080
Tuned Gradient Boosting 0.701 ± 0.055 0.660 ± 0.064        0.607 ± 0.110 0.627 ± 0.075 0.744 ± 0.065


In [19]:
import numpy as np
import pandas as pd
from scipy.stats import t
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as IMBPipeline

def load_bupa_data():
    """
    Load the BUPA Liver Disorders dataset cleanly without target leakage.
    """
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/liver-disorders/bupa.data"
    column_names = ['mcv', 'alkphos', 'sgpt', 'sgot', 'gammagt', 'drinks', 'selector']
    print("Fetching BUPA Liver Disorders dataset...")
    df = pd.read_csv(url, header=None, names=column_names)
    
    # Define binary target using the selector field (avoids leaking 'drinks')
    df['target'] = (df['selector'] == 1).astype(int)
    
    X = df.drop(columns=['selector', 'target'])
    y = df['target'].values
    return X, y

def corrected_resampled_t_test(scores_model_a, scores_model_b, n_train, n_test):
    """
    Implements the Nadeau-Bengio corrected resampled paired t-test.
    Adjusts for variance underestimation caused by overlapping folds in k-fold CV.
    """
    diff = np.array(scores_model_a) - np.array(scores_model_b)
    n_folds = len(diff)
    mean_diff = np.mean(diff)
    
    adapter = (1.0 / n_folds) + (n_test / n_train)
    var_diff = np.var(diff, ddof=1)
    
    if var_diff == 0:
        return 0.0, 1.0
        
    corrected_var = var_diff * adapter
    t_stat = mean_diff / np.sqrt(corrected_var)
    
    # Two-tailed p-value calculation
    p_val = 2.0 * (1.0 - t.cdf(abs(t_stat), df=n_folds - 1))
    return t_stat, p_val

def compute_pairwise_matrix(model_scores_dict, n_train, n_test):
    """
    Computes a pairwise p-value matrix across all models with Holm-Bonferroni correction.
    """
    models = list(model_scores_dict.keys())
    k = len(models)
    p_matrix = pd.DataFrame(np.ones((k, k)), index=models, columns=models)
    
    raw_p_vals = []
    comparisons = []
    
    for i in range(k):
        for j in range(i + 1, k):
            m1, m2 = models[i], models[j]
            _, p_val = corrected_resampled_t_test(
                model_scores_dict[m1], model_scores_dict[m2], n_train, n_test
            )
            raw_p_vals.append(p_val)
            comparisons.append((m1, m2))
            
    # Holm-Bonferroni Correction for Multiple Testing
    sorted_indices = np.argsort(raw_p_vals)
    m = len(raw_p_vals)
    adjusted_p_vals = np.zeros(m)
    
    for rank, idx in enumerate(sorted_indices):
        adjusted_p_vals[idx] = min(1.0, raw_p_vals[idx] * (m - rank))
        
    for idx, (m1, m2) in enumerate(comparisons):
        p_matrix.loc[m1, m2] = adjusted_p_vals[idx]
        p_matrix.loc[m2, m1] = adjusted_p_vals[idx]
        
    return p_matrix

def main():
    X, y = load_bupa_data()
    n_samples = len(X)
    n_train = int(n_samples * 0.8) # Approx train size for 5-fold CV (4/5ths)
    n_test = int(n_samples * 0.2)  # Approx test size for 5-fold CV (1/5th)
    
    print(f"Dataset shape: {n_samples} rows, {X.shape[1]} features.\n")
    
    # ==========================================
    # 1. EVALUATE LEAKAGE LADDER (SMOTE vs Safe Pipeline)
    # ==========================================
    print("--- 1. Leakage Ladder: SMOTE Inflation Analysis ---")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Safe leakage-free pipeline with SMOTE encapsulated inside fold
    safe_smote_pipeline = IMBPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('clf', RandomForestClassifier(random_state=42))
    ])
    safe_scores = cross_val_score(safe_smote_pipeline, X, y, cv=cv, scoring='roc_auc')
    
    # Global pre-split SMOTE leakage simulation
    leaked_scores = []
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        smote = SMOTE(random_state=42)
        X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
        
        clf = RandomForestClassifier(random_state=42)
        clf.fit(X_train_resampled, y_train_resampled)
        y_pred_proba = clf.predict_proba(X_test_scaled)[:, 1]
        leaked_scores.append(roc_auc_score(y_test, y_pred_proba))
        
    print(f"Leakage-Free SMOTE Pipeline (R0) ROC-AUC: {np.mean(safe_scores):.3f} ± {np.std(safe_scores):.3f}")
    print(f"Global Pre-Split SMOTE Leakage (R3) ROC-AUC: {np.mean(leaked_scores):.3f} ± {np.std(leaked_scores):.3f}\n")
    
    # ==========================================
    # 2. NESTED CV & FOLD SCORES FOR STATISTICAL TESTING
    # ==========================================
    print("--- 2. Nested CV & Pairwise Corrected t-Test Matrix ---")
    
    models = {
        "Logistic Regression": Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
        "SVM": Pipeline([('scaler', StandardScaler()), ('clf', SVC(probability=True, random_state=42))]),
        "Random Forest": Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(random_state=42))]),
        "Gradient Boosting": Pipeline([('scaler', StandardScaler()), ('clf', GradientBoostingClassifier(random_state=42))])
    }
    
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    model_fold_roc_auc = {}
    
    for name, pipeline in models.items():
        # If tuning, wrap in GridSearchCV, else evaluate directly
        if name in ["Random Forest", "Gradient Boosting"]:
            param_grid = {'clf__n_estimators': [50, 100], 'clf__max_depth': [3, 5]}
            search = GridSearchCV(pipeline, param_grid, cv=inner_cv, scoring='roc_auc', n_jobs=-1)
            eval_model = search
        else:
            eval_model = pipeline
            
        # Collect fold-level ROC-AUC scores for statistical testing
        fold_scores = []
        for train_idx, test_idx in outer_cv.split(X, y):
            X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]
            
            eval_model.fit(X_tr, y_tr)
            preds = eval_model.predict_proba(X_te)[:, 1]
            fold_scores.append(roc_auc_score(y_te, preds))
            
        model_fold_roc_auc[name] = fold_scores
        print(f"{name} Fold ROC-AUCs: {[round(s, 3) for s in fold_scores]} (Mean: {np.mean(fold_scores):.3f})")

    # Compute Pairwise Corrected t-test Matrix with Holm-Bonferroni correction
    p_matrix = compute_pairwise_matrix(model_fold_roc_auc, n_train, n_test)
    
    print("\n--- Holm-Bonferroni Adjusted p-value Matrix (ROC-AUC) ---")
    print(p_matrix.round(4).to_string())

if __name__ == "__main__":
    main()

Fetching BUPA Liver Disorders dataset...
Dataset shape: 345 rows, 6 features.

--- 1. Leakage Ladder: SMOTE Inflation Analysis ---
Leakage-Free SMOTE Pipeline (R0) ROC-AUC: 0.751 ± 0.082
Global Pre-Split SMOTE Leakage (R3) ROC-AUC: 0.751 ± 0.082

--- 2. Nested CV & Pairwise Corrected t-Test Matrix ---
Logistic Regression Fold ROC-AUCs: [np.float64(0.807), np.float64(0.637), np.float64(0.845), np.float64(0.641), np.float64(0.638)] (Mean: 0.714)
SVM Fold ROC-AUCs: [np.float64(0.821), np.float64(0.732), np.float64(0.854), np.float64(0.625), np.float64(0.651)] (Mean: 0.737)
Random Forest Fold ROC-AUCs: [np.float64(0.901), np.float64(0.662), np.float64(0.756), np.float64(0.655), np.float64(0.707)] (Mean: 0.736)
Gradient Boosting Fold ROC-AUCs: [np.float64(0.848), np.float64(0.748), np.float64(0.789), np.float64(0.677), np.float64(0.674)] (Mean: 0.747)

--- Holm-Bonferroni Adjusted p-value Matrix (ROC-AUC) ---
                     Logistic Regression     SVM  Random Forest  Gradient Boosting